# トランプ分割モデルの学習（Colab / CPU可）

このノートブックは、Google Drive にある**実写真363枚**（1枚に1カード）から
学習データを自動生成し、カードの**重なりを分離できる**セグメンテーションモデルを学習します。

**やること（上から順にセルを実行するだけ）**

1. Drive をマウント
2. 学習プログラムを書き出す
3. 写真からカードと背景を取り出す
4. 重なり込みの合成シーンを作る
5. 学習する（GPUが無くても動きます）
6. 精度を確認する
7. 重みファイルを Drive に保存する

最後に出力される `card_seg_unet.pt` を、リポジトリの
`services/recognition/models/` に置いてデプロイしてください。

**注意**: 学習に使うのは Drive の実写真だけです。アプリ表示用のデザインカード画像
（`apps/web/public/cards/`）は使いません。

## 1. Drive をマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 学習プログラムを書き出す

In [ ]:
# このセルが学習に必要なプログラムを /content に書き出します。
# （リポジトリの実物から自動生成しているので、中身は本番と同一です）
import json, pathlib, sys

FILES = json.loads(r'''{"card_splitter_v2.py": "\"\"\"Card detection v2 — the shared detection core for the app and the CLI.\n\nWhy this exists\n---------------\nThe original ``card_splitter_first.py`` finds cards by tracing Canny edges, and\n``card_splitter_revenge.py`` finds them by thresholding \"white\". Both fail in\nways that are structural, not tuning problems:\n\n* A white card on a bright wooden table has a *weak* outer edge, while a face\n  card's printed inner frame has a *strong* one — so the edge tracer returns the\n  inner frame and the crop is the middle of the K instead of the whole card.\n* Specular highlights on the table are white, and so is the card's own face, so\n  a \"whiteness\" threshold cannot separate them even in principle.\n* Two cards laid touching merge into one contour whose bounding box is\n  5.0 x 3.5 card-units — an aspect ratio of 1.428 against a single card's 1.400.\n  The two are 2% apart, so *no* aspect-ratio filter can tell them apart. The\n  merged blob sails through the filter and gets cropped as a single card.\n\nNeither program uses the two facts that make this problem easy: a playing card\nis a rounded rectangle with a fixed 2.5:3.5 aspect ratio, and every card in one\nphoto is the same physical size. This module is built around them.\n\nThe approach\n------------\n1. **Segment \"not the table\", not \"white\".** The table is a large, roughly\n   uniform region; sample it from the image border and segment by colour\n   distance from it, weighting chroma over lightness so glare doesn't matter.\n2. **Propose from several independent binarisations and union the results.**\n   No single threshold survives every lighting condition; a candidate only has\n   to be found by one of them.\n3. **Score candidates against the card prior** — aspect ratio, rectangularity,\n   and how much real image gradient actually sits under the quad's four edges.\n4. **Kill inner frames** by containment: a quad strictly inside a plausible card\n   quad is a printed frame, not a card.\n5. **Split merged blobs by rectifying first.** Warp the blob flat, and the seams\n   between touching cards become exactly axis-parallel lines that a projection\n   profile finds reliably. (This is what ``revenge.py`` attempted, but it ran the\n   projection on the raw perspective image where seams are neither straight nor\n   vertical.)\n6. **Enforce one card size per photo.** The median accepted card size rejects\n   both the too-small (inner frames) and the too-large (unsplit merges).\n\nEverything here is classical CV with no learned weights, so it runs anywhere\nOpenCV does. The optional learned engine (see ``card_seg_model.py``) plugs in as\na better *proposal* source and reuses stages 3-6 unchanged.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass, field\nfrom typing import Any, Sequence\n\nimport cv2\nimport numpy as np\n\n# A playing card is 2.5 x 3.5 inches: short / long.\nCARD_ASPECT = 2.5 / 3.5  # 0.7142857...\n\n# Crop sizes. The service normalises to 600x900 because the rank model's own\n# pipeline works at that size; the standalone annotation CLI emits 256x392 to\n# stay byte-compatible with the original script's output.\nCARD_W = 600\nCARD_H = 900\nPADDING_RATIO = 0.06\n\n# Detection runs on a downscaled copy — card edges are low-frequency, and this\n# keeps a 12MP phone photo inside the 0.5s budget on a 2-core shared CPU.\nWORK_MAX_SIDE = 1024\n\n# A card must cover at least this fraction of the frame to be considered. Five\n# cards in a wide shot each cover ~2%, so this is deliberately permissive.\nMIN_AREA_FRAC = 0.004\nMAX_AREA_FRAC = 0.92\n\nMAX_CARDS = 5\n\n\n# Overlap tests rasterise quads onto this canvas instead of the working image:\n# the answers are area ratios, which are scale-invariant, and it turns an O(n^2)\n# sweep over megapixel masks into bitwise ops on a few tens of kilobytes.\nOVERLAP_CANVAS = 256\n\n\n@dataclass\nclass CardQuad:\n    \"\"\"One detected card: its four corners in full-resolution image space.\"\"\"\n\n    corners: np.ndarray               # (4, 2) float32, ordered TL, TR, BR, BL\n    score: float = 0.0\n    sources: set[str] = field(default_factory=set)\n    aspect: float = 0.0               # short / long\n    extent: float = 0.0               # contour area / rotated-rect area\n    edge_support: float = 0.0\n    split_index: int | None = None    # set when produced by seam splitting\n    occluded: bool = False\n    _raster: np.ndarray | None = field(default=None, repr=False, compare=False)\n\n    @property\n    def area(self) -> float:\n        return float(cv2.contourArea(self.corners.astype(np.float32)))\n\n    @property\n    def center(self) -> np.ndarray:\n        return self.corners.mean(axis=0)\n\n    def dims(self) -> tuple[float, float]:\n        \"\"\"(short side, long side) in pixels, averaged over opposite edges.\"\"\"\n        tl, tr, br, bl = self.corners\n        a = 0.5 * (math.dist(tr, tl) + math.dist(br, bl))\n        b = 0.5 * (math.dist(bl, tl) + math.dist(br, tr))\n        return (min(a, b), max(a, b))\n\n\n# ---------------------------------------------------------------------------\n# Geometry helpers\n# ---------------------------------------------------------------------------\n\ndef order_points(pts: np.ndarray) -> np.ndarray:\n    \"\"\"Order 4 points as top-left, top-right, bottom-right, bottom-left.\n\n    Uses the angle around the centroid rather than the sum/difference trick from\n    the original script: the sum/difference rule mislabels corners once a card\n    is rotated past ~45 degrees, which silently produced 90-degree-wrong crops.\n    \"\"\"\n    pts = np.asarray(pts, dtype=np.float32).reshape(-1, 2)\n    center = pts.mean(axis=0)\n    angles = np.arctan2(pts[:, 1] - center[1], pts[:, 0] - center[0])\n    order = np.argsort(angles)\n    pts = pts[order]\n\n    # Rotate the (now counter-clockwise-from-+x) ring so it starts top-left.\n    sums = pts.sum(axis=1)\n    start = int(np.argmin(sums))\n    pts = np.roll(pts, -start, axis=0)\n\n    # Ensure clockwise TL -> TR -> BR -> BL.\n    if cv2.contourArea(pts.astype(np.float32)) < 0:\n        pts = pts[::-1]\n        start = int(np.argmin(pts.sum(axis=1)))\n        pts = np.roll(pts, -start, axis=0)\n    return pts.astype(np.float32)\n\n\ndef expand_quad(quad: np.ndarray, ratio: float = PADDING_RATIO) -> np.ndarray:\n    \"\"\"Grow a quad outward from its centre (keeps the original's 6% margin).\"\"\"\n    quad = np.asarray(quad, dtype=np.float32)\n    center = quad.mean(axis=0)\n    return (center + (quad - center) * (1.0 + ratio)).astype(np.float32)\n\n\ndef _raster(q: CardQuad, shape: tuple[int, int]) -> np.ndarray:\n    \"\"\"Small binary rasterisation of a quad, cached on the candidate.\"\"\"\n    if q._raster is None:\n        h, w = shape\n        s = OVERLAP_CANVAS / max(h, w)\n        ch, cw = max(1, int(h * s)), max(1, int(w * s))\n        m = np.zeros((ch, cw), np.uint8)\n        cv2.fillConvexPoly(m, (q.corners * s).astype(np.int32), 1)\n        q._raster = m\n    return q._raster\n\n\ndef _quad_iou(a: CardQuad, b: CardQuad, shape: tuple[int, int]) -> float:\n    \"\"\"Mask IoU of two quads. Exact for rotated shapes, unlike bbox IoU.\"\"\"\n    ma, mb = _raster(a, shape), _raster(b, shape)\n    union = int(np.count_nonzero(ma | mb))\n    if not union:\n        return 0.0\n    return int(np.count_nonzero(ma & mb)) / union\n\n\ndef _containment(inner: CardQuad, outer: CardQuad,\n                 shape: tuple[int, int]) -> float:\n    \"\"\"Fraction of `inner`'s area that lies inside `outer`.\"\"\"\n    mi, mo = _raster(inner, shape), _raster(outer, shape)\n    ai = int(np.count_nonzero(mi))\n    if ai == 0:\n        return 0.0\n    return int(np.count_nonzero(mi & mo)) / ai\n\n\ndef _aspect_score(aspect: float) -> float:\n    \"\"\"1.0 at the true card ratio, falling off smoothly. `aspect` = short/long.\"\"\"\n    if aspect <= 0:\n        return 0.0\n    err = abs(aspect - CARD_ASPECT) / CARD_ASPECT\n    return float(math.exp(-(err ** 2) / (2 * 0.11 ** 2)))\n\n\n# ---------------------------------------------------------------------------\n# Stage 1 — foreground hypotheses\n# ---------------------------------------------------------------------------\n\ndef _fill_holes(mask: np.ndarray) -> np.ndarray:\n    \"\"\"Fill interior holes so a card's printed pips don't punch through it.\"\"\"\n    filled = mask.copy()\n    contours, _ = cv2.findContours(filled, cv2.RETR_EXTERNAL,\n                                   cv2.CHAIN_APPROX_SIMPLE)\n    cv2.drawContours(filled, contours, -1, 255, thickness=cv2.FILLED)\n    return filled\n\n\ndef _clean(mask: np.ndarray, k: int = 5, close_iter: int = 2) -> np.ndarray:\n    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))\n    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=close_iter)\n    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)\n    return _fill_holes(mask)\n\n\ndef _background_distance(lab: np.ndarray) -> np.ndarray:\n    \"\"\"Distance from the table's colour, in [0, 255].\n\n    The table is sampled from a band around the image border and summarised with\n    a *median* so a card touching the edge doesn't drag the estimate. Chroma\n    (a*, b*) is weighted far above lightness (L*) because a specular highlight\n    blows out L* while leaving the hue of the surface underneath intact — that\n    single weighting is what makes this survive the glare that defeats a\n    whiteness threshold.\n    \"\"\"\n    h, w = lab.shape[:2]\n    band = max(4, int(min(h, w) * 0.06))\n    border = np.concatenate([\n        lab[:band].reshape(-1, 3),\n        lab[-band:].reshape(-1, 3),\n        lab[:, :band].reshape(-1, 3),\n        lab[:, -band:].reshape(-1, 3),\n    ])\n    bg = np.median(border, axis=0)\n\n    diff = lab.astype(np.float32) - bg.astype(np.float32)\n    chroma = np.sqrt(diff[..., 1] ** 2 + diff[..., 2] ** 2)\n    light = np.abs(diff[..., 0])\n    dist = chroma + 0.35 * light\n    return np.clip(dist, 0, 255).astype(np.uint8)\n\n\ndef _kmeans_background(lab: np.ndarray, k: int = 4) -> np.ndarray:\n    \"\"\"Cluster the image and treat border-dominant clusters as background.\n\n    Handles tables the median model can't summarise with one colour (a\n    two-tone mat, a strong shadow gradient) by letting each cluster vote: a\n    cluster that is over-represented in the border band relative to its share of\n    the whole image is part of the table.\n    \"\"\"\n    h, w = lab.shape[:2]\n    step = max(1, int(math.sqrt(h * w / 20000)))\n    sample = lab[::step, ::step].reshape(-1, 3).astype(np.float32)\n    # Same chroma-over-lightness weighting as above.\n    sample[:, 0] *= 0.35\n\n    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 12, 1.0)\n    _, _, centers = cv2.kmeans(sample, k, None, criteria, 2,\n                               cv2.KMEANS_PP_CENTERS)\n\n    # Label at quarter resolution and upsample: cards are large, low-frequency\n    # regions, so the cluster map costs 16x less to build and looks the same.\n    small = cv2.resize(lab, (max(1, w // 4), max(1, h // 4)),\n                       interpolation=cv2.INTER_AREA).astype(np.float32)\n    small[..., 0] *= 0.35\n    sh, sw = small.shape[:2]\n\n    # Assign per cluster rather than broadcasting an (H*W, k, 3) tensor, which\n    # would allocate hundreds of megabytes on a phone-sized photo.\n    best = np.full((sh, sw), np.inf, np.float32)\n    labels = np.zeros((sh, sw), np.int32)\n    for c in range(k):\n        d = np.linalg.norm(small - centers[c], axis=2)\n        closer = d < best\n        best[closer] = d[closer]\n        labels[closer] = c\n\n    band = max(2, int(min(sh, sw) * 0.06))\n    border_mask = np.zeros((sh, sw), bool)\n    border_mask[:band] = border_mask[-band:] = True\n    border_mask[:, :band] = border_mask[:, -band:] = True\n\n    total = sh * sw\n    border_total = int(border_mask.sum())\n    background = np.zeros((sh, sw), np.uint8)\n    for c in range(k):\n        member = labels == c\n        share = member.sum() / total\n        border_share = (member & border_mask).sum() / max(border_total, 1)\n        if share > 0 and border_share >= share:\n            background |= member.astype(np.uint8)\n\n    fg = ((1 - background) * 255).astype(np.uint8)\n    return cv2.resize(fg, (w, h), interpolation=cv2.INTER_NEAREST)\n\n\ndef _hypothesis_masks(bgr: np.ndarray) -> dict[str, np.ndarray]:\n    \"\"\"Several independent foreground guesses; a card only needs one to hit.\"\"\"\n    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)\n    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)\n    blur = cv2.GaussianBlur(gray, (5, 5), 0)\n    masks: dict[str, np.ndarray] = {}\n\n    # H1 — distance from the table colour.\n    dist = _background_distance(lab)\n    _, m = cv2.threshold(dist, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)\n    masks[\"bg_model\"] = _clean(m)\n\n    # H2 — table colour clusters.\n    try:\n        masks[\"bg_kmeans\"] = _clean(_kmeans_background(lab))\n    except cv2.error:\n        pass\n\n    # H3 — global Otsu on lightness. Wins when the table is dark.\n    _, m = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)\n    masks[\"otsu\"] = _clean(m)\n\n    # H4 — low saturation. Wins when the table is strongly coloured (green felt).\n    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)\n    sat = hsv[..., 1]\n    _, m = cv2.threshold(sat, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)\n    masks[\"low_sat\"] = _clean(m)\n\n    # H5 — the original Canny route, kept because it is the one that works when\n    # card and table are the same colour and only the shadow line separates them.\n    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))\n    eq = clahe.apply(gray)\n    eqb = cv2.GaussianBlur(eq, (5, 5), 0)\n    med = float(np.median(eqb))\n    edges = cv2.Canny(eqb, int(max(0, 0.66 * med)), int(min(255, 1.33 * med)))\n    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))\n    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=2)\n    edges = cv2.dilate(edges, kernel, iterations=1)\n    masks[\"canny\"] = _fill_holes(edges)\n\n    return masks\n\n\n# ---------------------------------------------------------------------------\n# Stage 2 — contours to quads\n# ---------------------------------------------------------------------------\n\ndef _quad_from_contour(cnt: np.ndarray) -> tuple[np.ndarray, float]:\n    \"\"\"Best 4-corner fit for a contour, plus its rectangularity.\n\n    Prefers a genuine 4-point polygon approximation (which follows perspective)\n    and falls back to the min-area rectangle when the outline is too ragged.\n    \"\"\"\n    peri = cv2.arcLength(cnt, True)\n    best: np.ndarray | None = None\n    for eps in (0.02, 0.03, 0.04, 0.05, 0.015, 0.06, 0.08):\n        approx = cv2.approxPolyDP(cnt, eps * peri, True)\n        if len(approx) == 4 and cv2.isContourConvex(approx):\n            best = approx.reshape(4, 2).astype(np.float32)\n            break\n\n    rect = cv2.minAreaRect(cnt)\n    box = cv2.boxPoints(rect).astype(np.float32)\n    rect_area = float(rect[1][0] * rect[1][1])\n    cnt_area = float(cv2.contourArea(cnt))\n    extent = cnt_area / rect_area if rect_area > 0 else 0.0\n\n    if best is not None:\n        quad_area = float(cv2.contourArea(best))\n        # Only trust the polygon if it explains the contour about as well as the\n        # rectangle does; a bad approximation collapses a corner.\n        if quad_area > 0 and abs(quad_area - cnt_area) / max(cnt_area, 1) < 0.25:\n            return order_points(best), extent\n\n    return order_points(box), extent\n\n\ndef _border_fraction(cnt: np.ndarray, shape: tuple[int, int],\n                     margin: int = 3) -> float:\n    \"\"\"Fraction of a contour's points that lie on the image border.\n\n    A threshold that picks the *table* rather than the cards returns one blob\n    whose outline is largely the image frame itself. Cards don't do that, so\n    this cheaply rejects inverted-polarity masks at the source — which matters\n    because such a blob would otherwise look like a container and swallow every\n    real card in the containment test.\n    \"\"\"\n    h, w = shape\n    pts = cnt.reshape(-1, 2)\n    if len(pts) == 0:\n        return 0.0\n    on = ((pts[:, 0] <= margin) | (pts[:, 0] >= w - 1 - margin)\n          | (pts[:, 1] <= margin) | (pts[:, 1] >= h - 1 - margin))\n    return float(on.mean())\n\n\ndef _candidates_from_mask(mask: np.ndarray, source: str,\n                          image_area: float) -> list[CardQuad]:\n    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,\n                                   cv2.CHAIN_APPROX_SIMPLE)\n    shape = mask.shape[:2]\n    out: list[CardQuad] = []\n    for cnt in contours:\n        area = cv2.contourArea(cnt)\n        if area < image_area * MIN_AREA_FRAC or area > image_area * MAX_AREA_FRAC:\n            continue\n        if _border_fraction(cnt, shape) > 0.30:\n            continue\n        quad, extent = _quad_from_contour(cnt)\n        short, long = CardQuad(quad).dims()\n        if long <= 1:\n            continue\n        aspect = short / long\n        # Keep anything from a single card up to five in a row; the seam splitter\n        # decides what it actually is.\n        if aspect < CARD_ASPECT / (MAX_CARDS + 0.6) or aspect > 1.02:\n            continue\n        if extent < 0.55:\n            continue\n        out.append(CardQuad(corners=quad, sources={source},\n                            aspect=aspect, extent=extent))\n    return out\n\n\n# ---------------------------------------------------------------------------\n# Stage 3 — scoring against the card prior\n# ---------------------------------------------------------------------------\n\n_EDGE_OFFSETS = np.array([-2.0, -1.0, 0.0, 1.0, 2.0], np.float32)\n\n\ndef _edge_support(grad: np.ndarray, thr: float, quad: np.ndarray,\n                  samples: int = 48) -> float:\n    \"\"\"How much real image gradient sits under the quad's four edges.\n\n    Samples across each edge along its normal and asks whether the image\n    actually steps there. A quad hallucinated by a threshold has no step under\n    it; a real card border does.\n    \"\"\"\n    if thr <= 0:\n        return 0.0\n    h, w = grad.shape[:2]\n    quad = quad.astype(np.float32)\n\n    bases = []\n    normals = []\n    for i in range(4):\n        p0, p1 = quad[i], quad[(i + 1) % 4]\n        seg = p1 - p0\n        length = float(np.linalg.norm(seg))\n        if length < 4:\n            continue\n        n = max(8, min(samples, int(length / 3)))\n        t = np.linspace(0.06, 0.94, n, dtype=np.float32)[:, None]\n        bases.append(p0 + seg * t)\n        normals.append(np.repeat(\n            (np.array([-seg[1], seg[0]], np.float32) / length)[None, :], n, 0))\n    if not bases:\n        return 0.0\n\n    base = np.concatenate(bases)                       # (N, 2)\n    normal = np.concatenate(normals)                   # (N, 2)\n    pts = base[None, :, :] + normal[None, :, :] * _EDGE_OFFSETS[:, None, None]\n    xs = np.clip(np.rint(pts[..., 0]).astype(np.int32), 0, w - 1)\n    ys = np.clip(np.rint(pts[..., 1]).astype(np.int32), 0, h - 1)\n    best = grad[ys, xs].max(axis=0)                    # (N,)\n    return float((best >= thr).mean())\n\n\ndef _refine_quad_edges(grad: np.ndarray, quad: np.ndarray,\n                       band_frac: float = 0.028) -> np.ndarray:\n    \"\"\"Snap a quad's four sides onto the strongest edge near each of them.\n\n    Thresholds put a border roughly where the card is; a few pixels of slop\n    survives, and glare can bias a whole side inward. Each side is re-fitted to\n    the gradient ridge beside it and the corners are recovered by intersecting\n    consecutive sides — which also squares up corners that a ragged mask\n    rounded off. The search band is deliberately narrow so the fit cannot jump\n    to the card's own printed content.\n    \"\"\"\n    h, w = grad.shape[:2]\n    quad = np.asarray(quad, dtype=np.float32)\n    short = min(CardQuad(quad).dims())\n    band = max(2.0, short * band_frac)\n    offsets = np.arange(-band, band + 1e-3, 1.0, dtype=np.float32)\n\n    lines: list[tuple[np.ndarray, np.ndarray]] = []\n    for i in range(4):\n        p0, p1 = quad[i], quad[(i + 1) % 4]\n        seg = p1 - p0\n        length = float(np.linalg.norm(seg))\n        if length < 8:\n            return quad\n        direction = seg / length\n        normal = np.array([-direction[1], direction[0]], np.float32)\n\n        n = max(12, min(40, int(length / 8)))\n        t = np.linspace(0.12, 0.88, n, dtype=np.float32)[:, None]\n        base = p0 + seg * t                                    # (n, 2)\n\n        pts = base[None, :, :] + normal[None, None, :] * offsets[:, None, None]\n        xs = np.clip(np.rint(pts[..., 0]).astype(np.int32), 0, w - 1)\n        ys = np.clip(np.rint(pts[..., 1]).astype(np.int32), 0, h - 1)\n        resp = grad[ys, xs]                                    # (len(offsets), n)\n\n        best_idx = np.argmax(resp, axis=0)\n        best_val = resp[best_idx, np.arange(resp.shape[1])]\n        d = offsets[best_idx]\n\n        # Keep only samples with a convincing ridge; if too few agree, leave the\n        # side where the mask put it rather than fitting to noise.\n        strong = best_val >= max(float(np.percentile(grad, 90)), 1e-6)\n        if strong.sum() < max(6, n // 3):\n            lines.append((p0.copy(), direction.copy()))\n            continue\n\n        # Reject outliers around the median offset before fitting.\n        med = float(np.median(d[strong]))\n        keep = strong & (np.abs(d - med) <= max(1.5, band * 0.6))\n        if keep.sum() < max(5, n // 4):\n            lines.append((p0 + normal * med, direction.copy()))\n            continue\n\n        fit_pts = base[keep] + normal * d[keep][:, None]\n        centroid = fit_pts.mean(axis=0)\n        _, _, vt = np.linalg.svd(fit_pts - centroid, full_matrices=False)\n        fitted_dir = vt[0].astype(np.float32)\n        if float(np.dot(fitted_dir, direction)) < 0:\n            fitted_dir = -fitted_dir\n        lines.append((centroid.astype(np.float32), fitted_dir))\n\n    corners = []\n    for i in range(4):\n        p_prev, d_prev = lines[(i - 1) % 4]\n        p_cur, d_cur = lines[i]\n        cross = float(d_prev[0] * d_cur[1] - d_prev[1] * d_cur[0])\n        if abs(cross) < 1e-3:            # near-parallel: no usable intersection\n            return quad\n        diff = p_cur - p_prev\n        t = (diff[0] * d_cur[1] - diff[1] * d_cur[0]) / cross\n        corners.append(p_prev + d_prev * t)\n\n    refined = np.asarray(corners, np.float32)\n    if not np.all(np.isfinite(refined)):\n        return quad\n    # A refinement should nudge, not redraw: reject a fit that moved a corner\n    # more than the search band could justify.\n    if float(np.max(np.linalg.norm(refined - quad, axis=1))) > band * 3.0:\n        return quad\n    return order_points(refined)\n\n\ndef _score_quad(q: CardQuad, grad: np.ndarray, thr: float) -> float:\n    q.edge_support = _edge_support(grad, thr, q.corners)\n    a = _aspect_score(q.aspect)\n    # Several independent binarisations agreeing is real evidence, so let the\n    # source count nudge the score — but cap it so it can't outweigh geometry.\n    agreement = min(len(q.sources), 3) / 3.0\n    q.score = float(0.42 * a + 0.24 * q.extent + 0.24 * q.edge_support\n                    + 0.10 * agreement)\n    return q.score\n\n\n# ---------------------------------------------------------------------------\n# Stage 4 — split blobs of touching cards, after rectifying them\n# ---------------------------------------------------------------------------\n\ndef _rectify(gray: np.ndarray, quad: np.ndarray,\n             out_w: int, out_h: int) -> tuple[np.ndarray, np.ndarray]:\n    dst = np.array([[0, 0], [out_w - 1, 0], [out_w - 1, out_h - 1],\n                    [0, out_h - 1]], np.float32)\n    m = cv2.getPerspectiveTransform(quad.astype(np.float32), dst)\n    return cv2.warpPerspective(gray, m, (out_w, out_h)), m\n\n\ndef _seam_profile(rect_gray: np.ndarray) -> np.ndarray:\n    \"\"\"Per-column \"is there a full-height edge here\" score, in [0, 1].\n\n    The seam where two cards touch runs the entire height of the pair; the\n    printed pips and indices that also throw gradients span only a fraction of\n    it. So the score is the *fraction of rows* in which the column carries a\n    strong gradient — near 1 over a seam, small over a pip. Using the fraction\n    rather than the mean is what keeps card artwork from reading as a seam.\n\n    A low quantile down each column expresses the same idea without needing a\n    threshold, and was tried; it measured worse on the benchmark, because a real\n    seam fades in and out along its length and any single low quantile is\n    dragged down by the rows where it fades.\n    \"\"\"\n    gx = np.abs(cv2.Scharr(rect_gray, cv2.CV_32F, 1, 0))\n    # Blur along both axes. Vertically, so a ridge interrupted by a pip still\n    # reads as continuous; horizontally, because the two card edges meeting at a\n    # seam put their gradient peaks on either side of it and leave the middle\n    # flat — a small horizontal blur merges them into one ridge over the seam.\n    gx = cv2.GaussianBlur(gx, (5, 9), 0)\n\n    sample = gx[::3, ::3]\n    thr = float(np.percentile(sample, 82))\n    if thr <= 1e-6:\n        # More than 82% of this region is flat — a plain white card face. Without\n        # a floor every pixel would count as strong and the profile would carry\n        # no information at all, so touching cards could never be split. Anchor\n        # to the strongest gradient present instead, but keep the anchor low:\n        # printed pips are darker than the thin shadow line between two cards,\n        # so a high anchor would exclude the very thing being looked for.\n        thr = 0.15 * float(np.percentile(sample, 99.0))\n    if thr <= 1e-6:\n        return np.zeros(rect_gray.shape[1], np.float32)\n\n    profile = (gx >= thr).astype(np.float32).mean(axis=0)\n    return cv2.GaussianBlur(profile.reshape(1, -1), (5, 1), 0).ravel()\n\n\ndef _card_count_hypotheses(short: float, long: float) -> list[tuple[int, bool]]:\n    \"\"\"Plausible (n, split_along_long) readings of a blob's dimensions.\n\n    Two cards side by side measure 5.0 x 3.5 card-units — aspect 1.428 — against\n    a single landscape card's 1.400. They are 2% apart, so both readings stay on\n    the table here and the seam evidence, not the ratio, decides.\n    \"\"\"\n    ratio = long / short if short > 0 else 0.0\n    out: list[tuple[int, bool]] = []\n    if ratio <= 0:\n        return out\n    for n in range(2, MAX_CARDS + 1):\n        # n cards stacked along the long side, each card standing upright.\n        if abs((ratio / n) - CARD_ASPECT) / CARD_ASPECT < 0.13:\n            out.append((n, True))\n        # n cards stacked along the long side, each lying on its long edge.\n        if abs((ratio * n) - 1.0 / CARD_ASPECT) / (1.0 / CARD_ASPECT) < 0.13:\n            out.append((n, True))\n    # De-duplicate while keeping order.\n    seen: set[tuple[int, bool]] = set()\n    uniq = []\n    for item in out:\n        if item not in seen:\n            seen.add(item)\n            uniq.append(item)\n    return uniq\n\n\ndef _find_seams(rect_gray: np.ndarray, n: int) -> tuple[list[int], float] | None:\n    \"\"\"Locate n-1 seams near their predicted positions; None if unconvincing.\"\"\"\n    w = rect_gray.shape[1]\n    profile = _seam_profile(rect_gray)\n    baseline = float(np.median(profile))\n    spread = float(np.percentile(profile, 75) - np.percentile(profile, 25)) or 1e-3\n\n    seams: list[int] = []\n    strengths: list[float] = []\n    window = max(6, int(w * 0.035))\n    for k in range(1, n):\n        target = int(round(w * k / n))\n        lo, hi = max(1, target - window), min(w - 1, target + window)\n        if hi <= lo:\n            return None\n        local = profile[lo:hi]\n        idx = int(np.argmax(local))\n        pos = lo + idx\n        peak = float(local[idx])\n        # A real seam is a full-height ridge standing well clear of the noise.\n        if peak < 0.55 or (peak - baseline) < 2.0 * spread:\n            return None\n        seams.append(pos)\n        strengths.append(peak)\n\n    return seams, float(np.mean(strengths))\n\n\ndef _split_quad_at(quad: np.ndarray, m_inv: np.ndarray, seams: Sequence[int],\n                   rect_w: int, rect_h: int) -> list[np.ndarray]:\n    \"\"\"Map seam columns in rectified space back to quads in image space.\"\"\"\n    bounds = [0, *seams, rect_w]\n    out: list[np.ndarray] = []\n    for i in range(len(bounds) - 1):\n        x0, x1 = bounds[i], bounds[i + 1]\n        if x1 - x0 < rect_w * 0.08:\n            continue\n        pts = np.array([[x0, 0], [x1 - 1, 0], [x1 - 1, rect_h - 1],\n                        [x0, rect_h - 1]], np.float32).reshape(-1, 1, 2)\n        mapped = cv2.perspectiveTransform(pts, m_inv).reshape(4, 2)\n        out.append(order_points(mapped))\n    return out\n\n\ndef _maybe_split(q: CardQuad, gray: np.ndarray,\n                 debug: list[dict[str, Any]] | None = None) -> list[CardQuad]:\n    \"\"\"Return q as-is, or the cards it turns out to be made of.\"\"\"\n    short, long = q.dims()\n    hypotheses = _card_count_hypotheses(short, long)\n    if not hypotheses:\n        return [q]\n\n    # Rectify with the long side horizontal so seams are vertical.\n    rect_h = 420\n    rect_w = int(round(rect_h * long / max(short, 1e-3)))\n    rect_w = max(64, min(rect_w, 2400))\n\n    # order_points gives TL,TR,BR,BL of the quad as drawn; rotate the corner\n    # order when the quad is taller than it is wide so the long axis lands on x.\n    corners = q.corners\n    tl, tr, br, bl = corners\n    if np.linalg.norm(tr - tl) < np.linalg.norm(bl - tl):\n        corners = np.array([tr, br, bl, tl], np.float32)\n\n    rect_gray, m = _rectify(gray, corners, rect_w, rect_h)\n    m_inv = np.linalg.inv(m)\n\n    best: tuple[float, list[int], int] | None = None\n    for n, _ in hypotheses:\n        found = _find_seams(rect_gray, n)\n        if found is None:\n            continue\n        seams, strength = found\n        # Prefer the strongest evidence; break ties toward fewer cards.\n        key = strength - 0.02 * n\n        if best is None or key > best[0]:\n            best = (key, seams, n)\n\n    if debug is not None:\n        debug.append({\n            \"hypotheses\": [h[0] for h in hypotheses],\n            \"chosen\": best[2] if best else 1,\n            \"profile\": _seam_profile(rect_gray).tolist(),\n            \"rect_size\": [rect_w, rect_h],\n        })\n\n    if best is None:\n        return [q]\n\n    _, seams, n = best\n    pieces = _split_quad_at(corners, m_inv, seams, rect_w, rect_h)\n    if len(pieces) < 2:\n        return [q]\n\n    out = []\n    for i, piece in enumerate(pieces):\n        sub = CardQuad(corners=piece, sources=set(q.sources) | {\"seam_split\"},\n                       extent=q.extent, split_index=i)\n        s, l = sub.dims()\n        sub.aspect = s / l if l > 0 else 0.0\n        out.append(sub)\n    return out\n\n\n# ---------------------------------------------------------------------------\n# Stage 5 — reconcile candidates into one card set\n# ---------------------------------------------------------------------------\n\n# A single card fills its own bounding rectangle almost completely. A blob of\n# several cards, or a threshold that leaked into the table, does not — which\n# makes extent the signal that tells \"a card\" from \"a region containing cards\".\nSINGLE_CARD_EXTENT = 0.85\n\n\ndef _merge_duplicates(cands: list[CardQuad],\n                      shape: tuple[int, int]) -> list[CardQuad]:\n    \"\"\"Collapse candidates that describe the same card, keeping the best one.\n\n    Two things are deliberately *not* merged. Overlapping-but-distinct quads are\n    left alone because cards really do overlap, and NMS that dropped them would\n    delete a card the player is holding. Nested quads are left alone too: a big\n    region and a card inside it can exceed the IoU threshold, and merging them by\n    score would let an over-segmented blob delete the very card it contains.\n    Nesting is settled later, on evidence rather than score.\n    \"\"\"\n    cands = sorted(cands, key=lambda c: c.score, reverse=True)\n    kept: list[CardQuad] = []\n    for c in cands:\n        duplicate_of = None\n        for k in kept:\n            if _quad_iou(c, k, shape) <= 0.55:\n                continue\n            inner, outer = (c, k) if c.area <= k.area else (k, c)\n            # Only a *meaningfully* bigger outer quad counts as nesting; two\n            # hypotheses outlining the same card differ by a few percent and\n            # must still collapse into one.\n            if (outer.area > inner.area * 1.25\n                    and _containment(inner, outer, shape) > 0.80):\n                continue  # nesting, not duplication\n            duplicate_of = k\n            break\n        if duplicate_of is None:\n            kept.append(c)\n        else:\n            duplicate_of.sources |= c.sources\n    return kept\n\n\ndef _drop_inner_frames(cands: list[CardQuad],\n                       shape: tuple[int, int]) -> list[CardQuad]:\n    \"\"\"Remove quads that sit wholly inside another plausible card.\n\n    This is the fix for the face-card failure: a K's printed frame is a strong,\n    clean rectangle *inside* the card outline, so edge strength alone prefers it.\n    Containment is the signal that settles it — a card is never inside a card.\n    \"\"\"\n    def looks_like_one_card(q: CardQuad) -> bool:\n        return q.extent >= SINGLE_CARD_EXTENT and _aspect_score(q.aspect) > 0.5\n\n    kept: list[CardQuad] = []\n    for c in cands:\n        c_area = c.area\n        drop = False\n        for other in cands:\n            if other is c or c_area <= 0:\n                continue\n\n            # Case 1 — c sits inside `other`. If `other` is itself a convincing\n            # single card, c is that card's printed frame. Bounding the ratio\n            # matters: a frame runs 50-85% of its card, so a genuine container is\n            # only modestly bigger, and without the bound an oversized blob could\n            # pose as the container and delete every real card inside it.\n            if (1.12 < other.area / c_area < 3.0\n                    and looks_like_one_card(other)\n                    and _containment(c, other, shape) > 0.90):\n                drop = True\n                break\n\n            # Case 2 — the mirror image: `other` is a convincing single card and\n            # c merely contains it while not looking like a card itself. That is\n            # an over-segmented region (a threshold that swallowed the table), so\n            # the region goes and the card it holds stays.\n            if (other.area < c_area / 1.12\n                    and looks_like_one_card(other)\n                    and not looks_like_one_card(c)\n                    and _containment(other, c, shape) > 0.85):\n                drop = True\n                break\n\n        if not drop:\n            kept.append(c)\n    return kept\n\n\ndef _unit_card_area(cands: list[CardQuad]) -> float:\n    \"\"\"Best guess at the area of one card in this photo, or 0 if unclear.\n\n    Every card in a photo is the same physical size and lies on the same plane,\n    so their areas cluster tightly. Estimating from the card-shaped, well-scoring\n    candidates only keeps the estimate away from junk blobs.\n    \"\"\"\n    good = [c.area for c in cands\n            if c.extent >= SINGLE_CARD_EXTENT\n            and _aspect_score(c.aspect) > 0.5 and c.area > 0]\n    if not good:\n        good = [c.area for c in cands\n                if _aspect_score(c.aspect) > 0.35 and c.area > 0]\n    if not good:\n        return 0.0\n    return float(np.median(good))\n\n\ndef _enforce_size_consistency(cands: list[CardQuad], unit: float\n                              ) -> list[CardQuad]:\n    \"\"\"Drop candidates whose size disagrees with the rest of the photo.\n\n    Rejects the residue both failure modes leave behind: inner frames come out\n    too small, and blobs that resisted splitting come out too large.\n    \"\"\"\n    if unit <= 0 or len(cands) < 2:\n        return cands\n    keep = [c for c in cands if 0.5 * unit <= c.area <= 1.9 * unit]\n    return keep if keep else cands\n\n\n# ---------------------------------------------------------------------------\n# Public API\n# ---------------------------------------------------------------------------\n\ndef detect_cards_classic(image: np.ndarray, *, max_cards: int = MAX_CARDS,\n                         debug: dict[str, Any] | None = None) -> list[CardQuad]:\n    \"\"\"Detect cards with no learned weights. Corners are in full-res coords.\"\"\"\n    h, w = image.shape[:2]\n    scale = min(1.0, WORK_MAX_SIDE / max(h, w))\n    work = (cv2.resize(image, (int(w * scale), int(h * scale)),\n                       interpolation=cv2.INTER_AREA) if scale < 1.0 else image)\n    wh, ww = work.shape[:2]\n    image_area = float(wh * ww)\n\n    gray = cv2.cvtColor(work, cv2.COLOR_BGR2GRAY)\n    grad = cv2.magnitude(cv2.Scharr(gray, cv2.CV_32F, 1, 0),\n                         cv2.Scharr(gray, cv2.CV_32F, 0, 1))\n    grad_thr = float(np.percentile(grad, 88))\n\n    masks = _hypothesis_masks(work)\n    candidates: list[CardQuad] = []\n    for name, mask in masks.items():\n        candidates.extend(_candidates_from_mask(mask, name, image_area))\n\n    if debug is not None:\n        debug[\"masks\"] = masks\n        debug[\"raw_candidates\"] = len(candidates)\n        debug[\"work_shape\"] = (wh, ww)\n\n    for c in candidates:\n        _score_quad(c, grad, grad_thr)\n\n    candidates = _merge_duplicates(candidates, (wh, ww))\n\n    # Split before sizing up the photo: a merged pair has to become two cards\n    # before the \"one card size per photo\" estimate can mean anything.\n    split_debug: list[dict[str, Any]] = []\n    expanded: list[CardQuad] = []\n    for c in candidates:\n        expanded.extend(_maybe_split(c, gray, split_debug))\n    for c in expanded:\n        if c.split_index is not None:\n            _score_quad(c, grad, grad_thr)\n    candidates = _merge_duplicates(expanded, (wh, ww))\n\n    unit = _unit_card_area(candidates)\n    candidates = _enforce_size_consistency(candidates, unit)\n    candidates = _drop_inner_frames(candidates, (wh, ww))\n    candidates = [c for c in candidates if c.score >= 0.45]\n\n    # Snap the survivors onto the real edges before they become crops.\n    for c in candidates:\n        c.corners = _refine_quad_edges(grad, c.corners)\n        c._raster = None\n        short, long = c.dims()\n        c.aspect = short / long if long > 0 else 0.0\n\n    candidates.sort(key=lambda c: c.score, reverse=True)\n    candidates = candidates[:max_cards]\n    # Left-to-right, matching the original script's ordering.\n    candidates.sort(key=lambda c: float(c.center[0]))\n\n    if scale < 1.0:\n        inv = 1.0 / scale\n        for c in candidates:\n            c.corners = (c.corners * inv).astype(np.float32)\n\n    if debug is not None:\n        debug[\"split\"] = split_debug\n        debug[\"unit_area\"] = unit\n        debug[\"final\"] = len(candidates)\n\n    return candidates\n\n\ndef warp_card(image: np.ndarray, quad: np.ndarray | CardQuad,\n              output_size: tuple[int, int] = (CARD_W, CARD_H)) -> np.ndarray:\n    \"\"\"Perspective-correct a card quad to an upright crop of `output_size`.\"\"\"\n    corners = quad.corners if isinstance(quad, CardQuad) else quad\n    rect = expand_quad(order_points(corners))\n    tl, tr, br, bl = rect\n\n    warp_w = max(int(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl))), 1)\n    warp_h = max(int(max(np.linalg.norm(br - tr), np.linalg.norm(bl - tl))), 1)\n\n    dst = np.array([[0, 0], [warp_w - 1, 0], [warp_w - 1, warp_h - 1],\n                    [0, warp_h - 1]], np.float32)\n    m = cv2.getPerspectiveTransform(rect, dst)\n    warped = cv2.warpPerspective(image, m, (warp_w, warp_h))\n\n    if warped.shape[1] > warped.shape[0]:\n        warped = cv2.rotate(warped, cv2.ROTATE_90_CLOCKWISE)\n\n    interp = (cv2.INTER_AREA if warped.shape[0] > output_size[1]\n              else cv2.INTER_CUBIC)\n    return cv2.resize(warped, output_size, interpolation=interp)\n\n\ndef classic_is_confident(quads: list[CardQuad]) -> bool:\n    \"\"\"Whether the classical result looks like a clean read of the scene.\n\n    Confident means: it found something, every quad is close to a card's true\n    proportions, they agree on a size, and they scored well. Cards laid out with\n    gaps produce exactly that; cards that overlap do not, because the geometry\n    the classical engine relies on isn't there.\n    \"\"\"\n    if not quads:\n        return False\n    for q in quads:\n        if not (0.62 <= q.aspect <= 0.82):\n            return False\n        if q.score < 0.55:\n            return False\n    if len(quads) > 1:\n        areas = sorted(q.area for q in quads)\n        if areas[0] <= 0 or areas[-1] / areas[0] > 1.5:\n            return False\n    return True\n\n\ndef detect_cards(image: np.ndarray, *, engine: str = \"auto\",\n                 max_cards: int = MAX_CARDS,\n                 debug: dict[str, Any] | None = None) -> list[CardQuad]:\n    \"\"\"Detect every card in `image`.\n\n    ``engine``:\n      ``classic``  geometry only, no weights.\n      ``learned``  the trained segmentation model (raises if unavailable).\n      ``auto``     classical first, and the learned model only to rescue the\n                   scenes the classical engine cannot do.\n\n    Why ``auto`` runs the classical engine first rather than preferring the\n    model: measured on held-out synthetic scenes, the learned engine is far\n    better where geometry fails — overlapping cards go from 16% to 48% recall,\n    a fanned hand from 1% to 46% — but it is *worse* on cards laid out with\n    gaps, which the classical engine already reads at ~90%. Always preferring\n    the model would trade away the common case to win the rare one. So the\n    classical answer is taken whenever it looks clean (see\n    ``classic_is_confident``), and the model is consulted only when it doesn't.\n\n    That ordering also keeps the usual request on the fast path: the classical\n    engine alone is well inside the latency budget, and the model's cost is\n    only paid for the frames that need it.\n    \"\"\"\n    if engine == \"classic\":\n        return detect_cards_classic(image, max_cards=max_cards, debug=debug)\n\n    from . import card_seg_model  # local import: torch is optional for the CLI\n\n    if engine == \"learned\":\n        return card_seg_model.detect_cards_learned(\n            image, max_cards=max_cards, debug=debug)\n\n    if engine != \"auto\":\n        raise ValueError(f\"unknown engine: {engine!r}\")\n\n    classic = detect_cards_classic(image, max_cards=max_cards, debug=debug)\n\n    if not card_seg_model.weights_available():\n        if debug is not None:\n            debug[\"engine\"] = \"classic\"\n            debug[\"fallback\"] = \"no weights\"\n        return classic\n\n    if classic_is_confident(classic):\n        if debug is not None:\n            debug[\"engine\"] = \"classic\"\n            debug[\"fallback\"] = \"classic confident\"\n        return classic\n\n    try:\n        learned = card_seg_model.detect_cards_learned(\n            image, max_cards=max_cards, debug=debug)\n    except Exception as exc:  # noqa: BLE001 - never break on a bad model\n        if debug is not None:\n            debug[\"engine\"] = \"classic\"\n            debug[\"fallback\"] = f\"learned failed: {exc}\"\n        return classic\n\n    if not learned:\n        if debug is not None:\n            debug[\"engine\"] = \"classic\"\n            debug[\"fallback\"] = \"learned found nothing\"\n        return classic\n\n    if debug is not None:\n        debug[\"engine\"] = \"learned\"\n        debug[\"fallback\"] = \"classic not confident\"\n    return learned\n\n\ndef split_cards(image: np.ndarray, *, engine: str = \"auto\",\n                output_size: tuple[int, int] = (CARD_W, CARD_H),\n                max_cards: int = MAX_CARDS,\n                debug: dict[str, Any] | None = None\n                ) -> tuple[list[np.ndarray], list[CardQuad]]:\n    \"\"\"Detect and crop in one call. Returns (crops, quads) in left-to-right order.\"\"\"\n    quads = detect_cards(image, engine=engine, max_cards=max_cards, debug=debug)\n    crops = [warp_card(image, q, output_size) for q in quads]\n    return crops, quads\n", "card_seg_model.py": "\"\"\"Learned card segmentation — the engine that handles overlapping cards.\n\nThe classical detector in ``card_splitter_v2`` reaches its ceiling when cards\noverlap: a card that is partly hidden has no closed outline to trace, and its\nvisible part is an L-shape that no rectangle filter accepts. Measured on\nsynthetic scenes built from held-out source photos, it recovers ~90% of\nwell-separated cards, ~51% of touching ones, and ~16% of overlapping ones.\n\nThis module fills that gap with a small U-Net (see\n``research/training/tiny_unet.py``) that predicts three classes per pixel —\nbackground, card interior, card border. Predicting the border explicitly is what\nseparates the instances: two cards sharing an edge form one connected region\nunder a binary card/not-card mask, but their *interiors* are disconnected once\nthe border is carved out, so ordinary connected components recovers them one by\none.\n\nThe interior of an occluded card is a partial shape, so each component is\ncompleted back to a full card rectangle before it is cropped — see\n``_complete_to_card``. That keeps every crop the shape of a whole card, which is\nwhat the existing suit and rank models were trained on, so nothing downstream\nhas to change to gain overlap support.\n\nThe weights are optional. If the file is missing the service runs the classical\nengine instead, so a checkout without the model still works.\n\nThe model is a *rescue*, not a replacement — ``detect_cards(engine=\"auto\")``\ntakes the classical answer whenever it looks clean and only consults this\nmodule otherwise. A model trained on too little source material beats geometry\non overlap while losing ground on the easy separated case, and the easy case is\nthe common one. ``research/training/card_seg_eval.py`` prints that comparison on\nheld-out material; check it before shipping a new set of weights.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport math\nimport os\nfrom pathlib import Path\nfrom typing import Any\n\nimport cv2\nimport numpy as np\n\ntry:  # inside the service package\n    from .card_splitter_v2 import (\n        CARD_ASPECT,\n        MAX_CARDS,\n        CardQuad,\n        _aspect_score,\n        _merge_duplicates,\n        _refine_quad_edges,\n        order_points,\n    )\nexcept ImportError:  # standing alone, e.g. next to the file on Colab\n    from card_splitter_v2 import (  # type: ignore\n        CARD_ASPECT,\n        MAX_CARDS,\n        CardQuad,\n        _aspect_score,\n        _merge_duplicates,\n        _refine_quad_edges,\n        order_points,\n    )\n\nMODELS_DIR = Path(__file__).resolve().parent.parent.parent / \"models\"\n# Overridable so the training notebook can score a freshly trained file, and so\n# a deployment can point at weights mounted outside the image.\nWEIGHTS_PATH = Path(os.environ.get(\"CARD_SEG_WEIGHTS\")\n                    or (MODELS_DIR / \"card_seg_unet.pt\"))\n\n\ndef set_weights_path(path: str | Path) -> None:\n    \"\"\"Point at a different weights file and drop any loaded model.\"\"\"\n    global WEIGHTS_PATH, _model, _load_failed\n    WEIGHTS_PATH = Path(path)\n    _model = None\n    _load_failed = False\n\n\n# Must match research/training/tiny_unet.py.\nINPUT_W = 256\nINPUT_H = 192\nMEAN = np.array([0.485, 0.456, 0.406], np.float32)\nSTD = np.array([0.229, 0.224, 0.225], np.float32)\n\nLABEL_INTERIOR = 1\nLABEL_BORDER = 2\n\n_model: Any = None\n_load_failed = False\n\n\ndef weights_available() -> bool:\n    return WEIGHTS_PATH.is_file()\n\n\ndef _load():\n    \"\"\"Load the TorchScript module once per process.\"\"\"\n    global _model, _load_failed\n    if _model is not None:\n        return _model\n    if _load_failed:\n        raise RuntimeError(\"card segmentation weights previously failed to load\")\n    if not WEIGHTS_PATH.is_file():\n        _load_failed = True\n        raise FileNotFoundError(\n            f\"学習済みモデルが見つかりません: {WEIGHTS_PATH}\\n\"\n            \"research/notebooks/card_seg_synth_train.ipynb を Colab で実行して \"\n            \"card_seg_unet.pt を作り、上記の場所に置いてください。\"\n            \"（--engine auto なら学習モデル無しでも古典的手法で動きます）\")\n    try:\n        import torch\n\n        model = torch.jit.load(str(WEIGHTS_PATH), map_location=\"cpu\")\n        model.eval()\n        _model = model\n        return _model\n    except Exception:\n        _load_failed = True\n        raise\n\n\ndef _predict(image: np.ndarray) -> np.ndarray:\n    \"\"\"Return the per-pixel class map at the network's own resolution.\"\"\"\n    import torch\n\n    model = _load()\n    resized = cv2.resize(image, (INPUT_W, INPUT_H), interpolation=cv2.INTER_AREA)\n    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0\n    rgb = (rgb - MEAN) / STD\n    tensor = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0)\n    with torch.no_grad():\n        logits = model(tensor)\n    return logits.argmax(1)[0].cpu().numpy().astype(np.uint8)\n\n\ndef _axis_vectors(angle_deg: float) -> tuple[np.ndarray, np.ndarray]:\n    \"\"\"Unit vectors along a rotated rectangle's width and height axes.\"\"\"\n    rad = math.radians(angle_deg)\n    along_w = np.array([math.cos(rad), math.sin(rad)], np.float32)\n    along_h = np.array([-math.sin(rad), math.cos(rad)], np.float32)\n    return along_w, along_h\n\n\ndef _occluded_side(label: np.ndarray, centre: np.ndarray, axis: np.ndarray,\n                   half_extent: float, probe: float) -> int:\n    \"\"\"Which end of an axis the card continues under: -1, +1, or 0 if unclear.\n\n    A card is cut short because another card lies on top of it, and that\n    neighbour is itself card pixels. So the strip just beyond the cut end is\n    full of interior/border labels, while the strip beyond a genuine card edge\n    is background. Comparing the two ends says which way to grow — growing\n    symmetrically instead would slide the reconstructed card sideways by half\n    the hidden width and hand the rank model the wrong corner.\n    \"\"\"\n    h, w = label.shape[:2]\n    scores = []\n    for sign in (-1.0, 1.0):\n        base = centre + axis * sign * half_extent\n        hits = total = 0\n        for step in np.linspace(1.0, probe, 6):\n            for lateral in np.linspace(-half_extent * 0.6, half_extent * 0.6, 7):\n                perp = np.array([-axis[1], axis[0]], np.float32)\n                p = base + axis * sign * step + perp * lateral\n                x, y = int(round(p[0])), int(round(p[1]))\n                if 0 <= x < w and 0 <= y < h:\n                    total += 1\n                    if label[y, x] != 0:\n                        hits += 1\n        scores.append(hits / total if total else 0.0)\n\n    minus, plus = scores\n    if abs(minus - plus) < 0.25:\n        return 0\n    return -1 if minus > plus else 1\n\n\ndef _complete_to_card(contour: np.ndarray,\n                      label: np.ndarray | None = None) -> np.ndarray | None:\n    \"\"\"Fit a full card rectangle to a possibly partial component.\n\n    An unoccluded card's component is already rectangular and the min-area\n    rectangle is the answer. An occluded one is an L or a band, and its min-area\n    rectangle is too small in whichever direction the neighbour cut into it. The\n    card's aspect ratio is known exactly, so the deficient side is grown back to\n    2.5:3.5 — the visible edges pin down the orientation and the two dimensions\n    that survive, and the ratio supplies the one that didn't. Which *end* to grow\n    from is decided by looking for the occluder (see ``_occluded_side``).\n    \"\"\"\n    rect = cv2.minAreaRect(contour)\n    (cx, cy), (rw, rh), angle = rect\n    if rw <= 1 or rh <= 1:\n        return None\n\n    short, long = min(rw, rh), max(rw, rh)\n    ratio = short / long\n    new_short, new_long = short, long\n\n    if ratio < CARD_ASPECT * 0.92:\n        new_short = long * CARD_ASPECT          # neighbour ate into the short side\n    elif ratio > CARD_ASPECT * 1.10:\n        new_long = short / CARD_ASPECT          # the long side was clipped\n\n    new_size = ((new_short, new_long) if rw <= rh else (new_long, new_short))\n    centre = np.array([cx, cy], np.float32)\n\n    grow_w = new_size[0] - rw\n    grow_h = new_size[1] - rh\n    if label is not None and (grow_w > 1e-3 or grow_h > 1e-3):\n        along_w, along_h = _axis_vectors(angle)\n        if grow_w > grow_h:\n            side = _occluded_side(label, centre, along_w, rw / 2,\n                                  max(grow_w, 2.0))\n            centre = centre + along_w * side * (grow_w / 2)\n        else:\n            side = _occluded_side(label, centre, along_h, rh / 2,\n                                  max(grow_h, 2.0))\n            centre = centre + along_h * side * (grow_h / 2)\n\n    box = cv2.boxPoints(((float(centre[0]), float(centre[1])), new_size,\n                         angle)).astype(np.float32)\n    return order_points(box)\n\n\ndef detect_cards_learned(image: np.ndarray, *, max_cards: int = MAX_CARDS,\n                         debug: dict[str, Any] | None = None) -> list[CardQuad]:\n    \"\"\"Detect cards, including overlapping ones, with the trained segmenter.\"\"\"\n    h, w = image.shape[:2]\n    label = _predict(image)\n\n    interior = (label == LABEL_INTERIOR).astype(np.uint8)\n    # Erode a little more: the network's border is a few pixels at 256x192, and\n    # instances that still touch after it would merge back into one component.\n    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))\n    seeds = cv2.morphologyEx(interior, cv2.MORPH_OPEN, k, iterations=1)\n\n    n_labels, comp = cv2.connectedComponents(seeds, connectivity=4)\n    scale_x, scale_y = w / INPUT_W, h / INPUT_H\n    min_area = (INPUT_W * INPUT_H) * 0.004\n\n    quads: list[CardQuad] = []\n    for i in range(1, n_labels):\n        blob = (comp == i).astype(np.uint8)\n        if int(blob.sum()) < min_area:\n            continue\n        contours, _ = cv2.findContours(blob, cv2.RETR_EXTERNAL,\n                                       cv2.CHAIN_APPROX_SIMPLE)\n        if not contours:\n            continue\n        cnt = max(contours, key=cv2.contourArea)\n        quad = _complete_to_card(cnt, label)\n        if quad is None:\n            continue\n\n        # Occlusion shows up as a component much smaller than the card it sits\n        # in; flag it so callers know the crop contains borrowed pixels.\n        rect_area = cv2.contourArea(quad.astype(np.float32))\n        occluded = rect_area > 0 and (cv2.contourArea(cnt) / rect_area) < 0.72\n\n        full = quad * np.array([scale_x, scale_y], np.float32)\n        q = CardQuad(corners=full.astype(np.float32),\n                     sources={\"unet\"}, occluded=occluded)\n        short, long = q.dims()\n        q.aspect = short / long if long > 0 else 0.0\n        q.extent = float(cv2.contourArea(cnt) / rect_area) if rect_area else 0.0\n        q.score = 0.55 + 0.35 * _aspect_score(q.aspect)\n        quads.append(q)\n\n    quads = _merge_duplicates(quads, (h, w))\n    quads = [q for q in quads if _aspect_score(q.aspect) > 0.25]\n\n    # Snap onto real edges: the mask comes back at 256x192, so its outline is\n    # several full-resolution pixels thick before this runs.\n    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)\n    grad = cv2.magnitude(cv2.Scharr(gray, cv2.CV_32F, 1, 0),\n                         cv2.Scharr(gray, cv2.CV_32F, 0, 1))\n    for q in quads:\n        q.corners = _refine_quad_edges(grad, q.corners)\n        q._raster = None\n        short, long = q.dims()\n        q.aspect = short / long if long > 0 else 0.0\n\n    quads.sort(key=lambda q: q.score, reverse=True)\n    quads = quads[:max_cards]\n    quads.sort(key=lambda q: float(q.center[0]))\n\n    if debug is not None:\n        debug[\"engine\"] = \"learned\"\n        debug[\"label_map\"] = label\n        debug[\"components\"] = n_labels - 1\n        debug[\"final\"] = len(quads)\n\n    return quads\n", "card_synth.py": "\"\"\"Synthesise labelled multi-card scenes from single-card photos.\n\nWhy synthesise\n--------------\nSeparating cards that touch or overlap needs a learned segmenter, and training\none needs images labelled at pixel level. Hand-labelling those is exactly the\nwork this project is trying to remove, and the dataset we have is 363 photos of\n*one* card each — no overlap to learn from.\n\nThe way out is to build the training set out of the real photos. Every\nsingle-card photo yields two things the classical detector can extract reliably\n(one well-separated card is its easy case): the card, flattened, and the table\nit was lying on with the card painted out. Recombining those gives scenes with\nreal card printing, real table texture and real lighting, arranged into the\ntouching and overlapping layouts the dataset lacks — and because we place the\ncards ourselves, every pixel of the label is exact and free.\n\nNothing here uses the app's display artwork; the source material is only ever\nthe photographs.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Iterable, Sequence\n\nimport cv2\nimport numpy as np\n\n# Flattened source cards are kept at this size: big enough that a scene card is\n# usually downscaled (which hides resampling softness) and small enough to hold\n# a few hundred of them in memory at once.\nSOURCE_CARD_W = 320\nSOURCE_CARD_H = 448\n\n# Corner radius of a real card, as a fraction of its short side.\nCORNER_RADIUS_FRAC = 0.055\n\nCARD_ASPECT = 2.5 / 3.5\n\n# Label ids for the segmentation target.\nLABEL_BACKGROUND = 0\nLABEL_INTERIOR = 1\nLABEL_BORDER = 2\n\n\n@dataclass\nclass SceneCard:\n    \"\"\"One card placed in a synthetic scene.\"\"\"\n\n    quad: np.ndarray          # (4, 2) float32 corners in scene coordinates\n    visible: np.ndarray       # uint8 mask of the parts not covered by later cards\n    full: np.ndarray          # uint8 mask of the whole card, occlusion ignored\n\n    @property\n    def visible_fraction(self) -> float:\n        total = int(np.count_nonzero(self.full))\n        if total == 0:\n            return 0.0\n        return int(np.count_nonzero(self.visible)) / total\n\n\n@dataclass\nclass Scene:\n    image: np.ndarray\n    cards: list[SceneCard]\n\n    def label_map(self, border_px: int = 3) -> np.ndarray:\n        return build_label_map(self.cards, self.image.shape[:2], border_px)\n\n\n# ---------------------------------------------------------------------------\n# Source material\n# ---------------------------------------------------------------------------\n\ndef rounded_card_alpha(w: int, h: int,\n                       radius_frac: float = CORNER_RADIUS_FRAC) -> np.ndarray:\n    \"\"\"Alpha matte with a real card's rounded corners, anti-aliased.\"\"\"\n    ss = 4  # supersample, then average down for a soft edge\n    W, H = w * ss, h * ss\n    r = int(round(min(W, H) * radius_frac))\n    m = np.zeros((H, W), np.uint8)\n    cv2.rectangle(m, (r, 0), (W - r, H), 255, -1)\n    cv2.rectangle(m, (0, r), (W, H - r), 255, -1)\n    for cx, cy in ((r, r), (W - r, r), (r, H - r), (W - r, H - r)):\n        cv2.circle(m, (cx, cy), r, 255, -1)\n    return cv2.resize(m, (w, h), interpolation=cv2.INTER_AREA)\n\n\ndef _splitter():\n    \"\"\"Import the detector, whether we're in the repo or standing alone.\n\n    In the repo it lives with the recognition service so the app and the tools\n    can never drift apart. On Colab the notebook drops the same file next to\n    this one, so a plain import finds it there instead.\n    \"\"\"\n    import sys\n\n    try:\n        import card_splitter_v2 as v2  # type: ignore\n        return v2\n    except ImportError:\n        pass\n    service = Path(__file__).resolve().parents[2] / \"services\" / \"recognition\"\n    if str(service) not in sys.path:\n        sys.path.insert(0, str(service))\n    from app.recognition import card_splitter_v2 as v2  # type: ignore\n    return v2\n\n\ndef extract_card_and_plate(photo: np.ndarray) -> tuple[np.ndarray, \"Plate\"] | None:\n    \"\"\"Split a single-card photo into (flattened card, table with card removed).\n\n    Returns None when the detector doesn't find exactly one convincing card, so\n    a bad source photo drops out of the dataset instead of poisoning it.\n    \"\"\"\n    v2 = _splitter()\n    quads = v2.detect_cards_classic(photo, max_cards=2)\n    if len(quads) != 1:\n        return None\n    quad = quads[0]\n    short, long = quad.dims()\n    if long <= 0 or not (0.55 <= short / long <= 0.92):\n        return None\n\n    card = v2.warp_card(photo, quad, (SOURCE_CARD_W, SOURCE_CARD_H))\n\n    # Paint the card out of the photo so what remains is pure table. The mask is\n    # dilated well past the card so the drop shadow goes with it — a shadow left\n    # behind would be baked into every scene built on this plate.\n    h, w = photo.shape[:2]\n    mask = np.zeros((h, w), np.uint8)\n    grown = v2.expand_quad(quad.corners, ratio=0.22)\n    cv2.fillConvexPoly(mask, grown.astype(np.int32), 255)\n    plate = cv2.inpaint(photo, mask, 9, cv2.INPAINT_TELEA)\n\n    return card, Plate(image=plate, hole=mask)\n\n\n@dataclass\nclass Plate:\n    \"\"\"A photographed table surface, and where the card used to be.\n\n    Inpainting leaves a smear where the card was removed. Keeping the hole mask\n    lets scene composition crop *around* it, so scenes are built on genuine\n    table texture instead of on the repair.\n    \"\"\"\n\n    image: np.ndarray\n    hole: np.ndarray\n\n    def sample(self, w: int, h: int, rng: np.random.Generator,\n               tries: int = 12) -> np.ndarray:\n        ph, pw = self.image.shape[:2]\n        want = w / h\n        best: np.ndarray | None = None\n        best_cost = np.inf\n        for _ in range(tries):\n            ch = int(rng.uniform(0.5, 1.0) * ph)\n            cw = int(ch * want)\n            if cw > pw:\n                cw = pw\n                ch = int(cw / want)\n            if ch < 16 or cw < 16:\n                continue\n            y = int(rng.integers(0, max(1, ph - ch + 1)))\n            x = int(rng.integers(0, max(1, pw - cw + 1)))\n            cost = float(self.hole[y:y + ch, x:x + cw].mean())\n            if cost < best_cost:\n                best_cost = cost\n                best = self.image[y:y + ch, x:x + cw]\n                if cost < 1.0:\n                    break\n        if best is None:\n            best = self.image\n        return cv2.resize(best, (w, h), interpolation=cv2.INTER_AREA)\n\n\ndef load_sources(photo_paths: Iterable[Path], *, limit: int | None = None,\n                 progress: bool = True) -> tuple[list[np.ndarray], list[Plate]]:\n    \"\"\"Build the card and background-plate pools from single-card photos.\"\"\"\n    cards: list[np.ndarray] = []\n    plates: list[Plate] = []\n    paths = list(photo_paths)\n    if limit is not None:\n        paths = paths[:limit]\n    for i, p in enumerate(paths):\n        img = cv2.imread(str(p))\n        if img is None:\n            continue\n        got = extract_card_and_plate(img)\n        if got is None:\n            continue\n        card, plate = got\n        cards.append(card)\n        plates.append(plate)\n        if progress and (i + 1) % 25 == 0:\n            print(f\"  {i + 1}/{len(paths)} photos -> {len(cards)} cards\")\n    return cards, plates\n\n\n# ---------------------------------------------------------------------------\n# Photometric effects\n# ---------------------------------------------------------------------------\n\ndef _apply_shading(card: np.ndarray, rng: np.random.Generator) -> np.ndarray:\n    \"\"\"A soft linear light gradient across the card.\"\"\"\n    h, w = card.shape[:2]\n    angle = rng.uniform(0, 2 * math.pi)\n    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)\n    ramp = (math.cos(angle) * xx / w) + (math.sin(angle) * yy / h)\n    ramp = (ramp - ramp.min()) / (float(ramp.max() - ramp.min()) + 1e-6)\n    strength = rng.uniform(0.06, 0.30)\n    gain = (1.0 - strength / 2) + ramp * strength\n    return np.clip(card.astype(np.float32) * gain[..., None], 0, 255).astype(np.uint8)\n\n\ndef _apply_glare(card: np.ndarray, rng: np.random.Generator) -> np.ndarray:\n    \"\"\"Blow out an elliptical patch, the way a ceiling light reflects off gloss.\n\n    This is the failure the whiteness-threshold splitter could never survive, so\n    the segmenter has to meet it constantly during training.\n    \"\"\"\n    h, w = card.shape[:2]\n    spot = np.zeros((h, w), np.float32)\n    for _ in range(rng.integers(1, 3)):\n        cx = int(rng.uniform(0, w))\n        cy = int(rng.uniform(0, h))\n        ax = int(rng.uniform(w * 0.12, w * 0.55))\n        ay = int(rng.uniform(h * 0.08, h * 0.40))\n        cv2.ellipse(spot, (cx, cy), (ax, ay),\n                    float(rng.uniform(0, 180)), 0, 360, 1.0, -1)\n    spot = cv2.GaussianBlur(spot, (0, 0), sigmaX=max(w, h) * 0.06)\n    strength = rng.uniform(60, 165)\n    out = card.astype(np.float32) + spot[..., None] * strength\n    return np.clip(out, 0, 255).astype(np.uint8)\n\n\ndef _perspective_quad(w: float, h: float, rng: np.random.Generator) -> np.ndarray:\n    \"\"\"Corners of a card seen from a slightly off-axis camera.\"\"\"\n    jitter = min(w, h) * rng.uniform(0.0, 0.085)\n    base = np.array([[0, 0], [w, 0], [w, h], [0, h]], np.float32)\n    return base + rng.uniform(-jitter, jitter, size=(4, 2)).astype(np.float32)\n\n\ndef _rotate(points: np.ndarray, deg: float) -> np.ndarray:\n    rad = math.radians(deg)\n    c, s = math.cos(rad), math.sin(rad)\n    r = np.array([[c, -s], [s, c]], np.float32)\n    return points @ r.T\n\n\n# ---------------------------------------------------------------------------\n# Scene composition\n# ---------------------------------------------------------------------------\n\ndef _layout_positions(n: int, card_w: float, card_h: float,\n                      rng: np.random.Generator,\n                      mode: str) -> list[tuple[np.ndarray, float]]:\n    \"\"\"Centres and rotations for n cards under one of the layout styles.\"\"\"\n    out: list[tuple[np.ndarray, float]] = []\n\n    if mode == \"row_gap\":\n        gap = card_w * rng.uniform(0.14, 0.5)\n    elif mode == \"row_touch\":\n        # Edges within a hair of each other — the case a merged contour ruins.\n        gap = card_w * rng.uniform(-0.01, 0.02)\n    elif mode == \"row_overlap\":\n        gap = -card_w * rng.uniform(0.18, 0.55)\n    else:  # \"fan\" — the way hole cards are actually held\n        gap = -card_w * rng.uniform(0.30, 0.62)\n\n    total = n * card_w + (n - 1) * gap\n    x = -total / 2 + card_w / 2\n    base_tilt = rng.uniform(-14, 14)\n    for i in range(n):\n        if mode == \"fan\":\n            rot = base_tilt + (i - (n - 1) / 2) * rng.uniform(6, 17)\n            dy = abs(i - (n - 1) / 2) * card_h * rng.uniform(0.0, 0.05)\n        else:\n            rot = base_tilt + rng.uniform(-5, 5)\n            dy = rng.uniform(-card_h * 0.05, card_h * 0.05)\n        out.append((np.array([x, dy], np.float32), rot))\n        x += card_w + gap\n    return out\n\n\ndef compose_scene(cards: Sequence[np.ndarray], plates: Sequence[Plate],\n                  rng: np.random.Generator, *,\n                  size: tuple[int, int] = (640, 480),\n                  n_cards: int | None = None,\n                  mode: str | None = None) -> Scene:\n    \"\"\"Build one labelled scene. `size` is (width, height).\"\"\"\n    W, H = size\n    plate = plates[int(rng.integers(len(plates)))]\n    bg = plate.sample(W, H, rng)\n    if rng.random() < 0.5:\n        bg = cv2.flip(bg, int(rng.integers(-1, 2)))\n\n    n = int(n_cards if n_cards is not None else rng.integers(2, 6))\n    mode = mode or str(rng.choice([\"row_gap\", \"row_touch\", \"row_overlap\", \"fan\"],\n                                  p=[0.40, 0.24, 0.18, 0.18]))\n\n    # Size the cards so n of them comfortably fit the frame.\n    max_w = W / (n * 0.85 + 0.6)\n    card_w = float(rng.uniform(max_w * 0.55, max_w))\n    card_w = min(card_w, H * CARD_ASPECT * 0.82)\n    card_h = card_w / CARD_ASPECT\n\n    placements = _layout_positions(n, card_w, card_h, rng, mode)\n    scene_center = np.array([W / 2, H / 2], np.float32)\n    scene_center += rng.uniform(-0.06, 0.06, size=2).astype(np.float32) * [W, H]\n\n    canvas = bg.copy()\n    fulls: list[np.ndarray] = []\n    quads: list[np.ndarray] = []\n\n    for i, (offset, rot) in enumerate(placements):\n        src = cards[int(rng.integers(len(cards)))]\n        src = cv2.resize(src, (int(card_w * 1.4), int(card_h * 1.4)),\n                         interpolation=cv2.INTER_AREA)\n        if rng.random() < 0.5:\n            src = cv2.rotate(src, cv2.ROTATE_180)\n        src = _apply_shading(src, rng)\n        if rng.random() < 0.45:\n            src = _apply_glare(src, rng)\n        alpha = rounded_card_alpha(src.shape[1], src.shape[0])\n\n        local = _perspective_quad(card_w, card_h, rng)\n        local -= local.mean(axis=0)\n        dst = _rotate(local, rot) + scene_center + offset\n\n        sh, sw = src.shape[:2]\n        src_quad = np.array([[0, 0], [sw, 0], [sw, sh], [0, sh]], np.float32)\n        m = cv2.getPerspectiveTransform(src_quad, dst.astype(np.float32))\n        warped = cv2.warpPerspective(src, m, (W, H), flags=cv2.INTER_LINEAR)\n        warped_a = cv2.warpPerspective(alpha, m, (W, H), flags=cv2.INTER_LINEAR)\n\n        full = (warped_a > 127).astype(np.uint8)\n        if int(full.sum()) < 200:\n            continue\n\n        # Drop shadow, offset the way a card lifts slightly off the table.\n        shadow = cv2.GaussianBlur(warped_a.astype(np.float32) / 255.0, (0, 0),\n                                  sigmaX=max(3.0, card_w * 0.035))\n        sx, sy = int(card_w * rng.uniform(0.01, 0.05)), int(card_h * rng.uniform(0.01, 0.05))\n        shadow = np.roll(np.roll(shadow, sy, axis=0), sx, axis=1)\n        shadow *= rng.uniform(0.15, 0.42)\n        canvas = np.clip(canvas.astype(np.float32) * (1.0 - shadow[..., None]),\n                         0, 255).astype(np.uint8)\n\n        a = (warped_a.astype(np.float32) / 255.0)[..., None]\n        canvas = np.clip(canvas.astype(np.float32) * (1 - a)\n                         + warped.astype(np.float32) * a, 0, 255).astype(np.uint8)\n\n        fulls.append(full)\n        quads.append(dst.astype(np.float32))\n\n    # Later cards occlude earlier ones; recover what each one still shows.\n    scene_cards: list[SceneCard] = []\n    for i, (full, quad) in enumerate(zip(fulls, quads)):\n        covered = np.zeros_like(full)\n        for later in fulls[i + 1:]:\n            covered |= later\n        visible = (full & (1 - covered)).astype(np.uint8)\n        card = SceneCard(quad=quad, visible=visible, full=full)\n        # A card buried past recognition isn't a label, it's noise.\n        if card.visible_fraction >= 0.18:\n            scene_cards.append(card)\n\n    canvas = _finish(canvas, rng)\n    return Scene(image=canvas, cards=scene_cards)\n\n\ndef _finish(img: np.ndarray, rng: np.random.Generator) -> np.ndarray:\n    \"\"\"Whole-frame camera effects: exposure, white balance, blur, noise, JPEG.\"\"\"\n    out = img.astype(np.float32)\n    out *= rng.uniform(0.72, 1.24)                       # exposure\n    out *= rng.uniform(0.94, 1.06, size=3)               # white balance\n    out = np.clip(out, 0, 255).astype(np.uint8)\n\n    if rng.random() < 0.55:\n        k = int(rng.integers(1, 3)) * 2 + 1\n        out = cv2.GaussianBlur(out, (k, k), 0)\n    if rng.random() < 0.6:\n        noise = rng.normal(0, rng.uniform(1.5, 7.0), out.shape)\n        out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)\n    if rng.random() < 0.6:\n        q = int(rng.integers(45, 92))\n        ok, enc = cv2.imencode(\".jpg\", out, [cv2.IMWRITE_JPEG_QUALITY, q])\n        if ok:\n            out = cv2.imdecode(enc, cv2.IMREAD_COLOR)\n    return out\n\n\ndef build_label_map(cards: Sequence[SceneCard], shape: tuple[int, int],\n                    border_px: int = 3) -> np.ndarray:\n    \"\"\"Three-class target: background, card interior, card border.\n\n    The border class is what makes touching cards separable. Predicting \"card\"\n    alone would merge two cards sharing an edge into one blob — the very failure\n    being fixed. Carving a thin border out of every instance leaves the interiors\n    disconnected, so plain connected components recovers them one by one.\n    \"\"\"\n    h, w = shape\n    label = np.zeros((h, w), np.uint8)\n    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,\n                                  (border_px * 2 + 1, border_px * 2 + 1))\n    borders = np.zeros((h, w), np.uint8)\n    interiors = np.zeros((h, w), np.uint8)\n    for c in cards:\n        vis = c.visible\n        eroded = cv2.erode(vis, k, iterations=1)\n        borders |= (vis & (1 - eroded)).astype(np.uint8)\n        interiors |= eroded\n    label[interiors > 0] = LABEL_INTERIOR\n    label[borders > 0] = LABEL_BORDER\n    return label\n\n\ndef generate_dataset(cards: Sequence[np.ndarray], plates: Sequence[Plate],\n                     count: int, *, seed: int = 0,\n                     size: tuple[int, int] = (640, 480),\n                     border_px: int = 3,\n                     progress: bool = True):\n    \"\"\"Yield (image, label_map, scene) tuples.\"\"\"\n    rng = np.random.default_rng(seed)\n    for i in range(count):\n        scene = compose_scene(cards, plates, rng, size=size)\n        if not scene.cards:\n            continue\n        yield scene.image, scene.label_map(border_px), scene\n        if progress and (i + 1) % 200 == 0:\n            print(f\"  generated {i + 1}/{count}\")\n", "tiny_unet.py": "\"\"\"A deliberately small U-Net that separates touching and overlapping cards.\n\nDesign constraints, and what they buy\n-------------------------------------\n* **Trains on a CPU.** Colab may not hand out a GPU, so the network is sized so\n  a few thousand synthetic scenes converge in a sitting on two cores: ~0.39M\n  parameters at the default width, 256x192 input.\n* **Runs inside the 0.5s request budget** on the two shared cores the\n  recognition service gets, alongside torch already being loaded for SuitCNN.\n* **No new dependencies and no licence entanglement.** Plain ``torch.nn``, so\n  none of the AGPL obligations that come with the off-the-shelf YOLO packages —\n  which matters because this app is headed for commercial release.\n\nThe output is three classes per pixel — background, card interior, card border —\nrather than a plain card/not-card mask. Two cards sharing an edge produce one\nconnected blob under a binary mask, which is the exact failure being fixed;\npredicting the border explicitly carves a gap between them, so their interiors\ncome apart under ordinary connected components. The same trick recovers a card\nthat is partly hidden behind another, because the occluding card's border runs\nstraight through the seam.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nNUM_CLASSES = 3          # background, interior, border\nINPUT_W = 256\nINPUT_H = 192\n\n# Fed to the network; also what inference must reproduce exactly.\nMEAN = (0.485, 0.456, 0.406)\nSTD = (0.229, 0.224, 0.225)\n\n\ndef _block(cin: int, cout: int) -> nn.Sequential:\n    \"\"\"Two 3x3 convolutions. BatchNorm keeps CPU training stable at low width.\"\"\"\n    return nn.Sequential(\n        nn.Conv2d(cin, cout, 3, padding=1, bias=False),\n        nn.BatchNorm2d(cout),\n        nn.ReLU(inplace=True),\n        nn.Conv2d(cout, cout, 3, padding=1, bias=False),\n        nn.BatchNorm2d(cout),\n        nn.ReLU(inplace=True),\n    )\n\n\nclass TinyUNet(nn.Module):\n    \"\"\"Four-level U-Net, ~0.39M parameters at the default width.\"\"\"\n\n    def __init__(self, width: int = 16, num_classes: int = NUM_CLASSES):\n        super().__init__()\n        w1, w2, w3, w4 = width, width * 2, width * 4, width * 6\n\n        self.enc1 = _block(3, w1)\n        self.enc2 = _block(w1, w2)\n        self.enc3 = _block(w2, w3)\n        self.bottleneck = _block(w3, w4)\n\n        self.up3 = nn.ConvTranspose2d(w4, w3, 2, stride=2)\n        self.dec3 = _block(w3 * 2, w3)\n        self.up2 = nn.ConvTranspose2d(w3, w2, 2, stride=2)\n        self.dec2 = _block(w2 * 2, w2)\n        self.up1 = nn.ConvTranspose2d(w2, w1, 2, stride=2)\n        self.dec1 = _block(w1 * 2, w1)\n\n        self.head = nn.Conv2d(w1, num_classes, 1)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        e1 = self.enc1(x)\n        e2 = self.enc2(F.max_pool2d(e1, 2))\n        e3 = self.enc3(F.max_pool2d(e2, 2))\n        b = self.bottleneck(F.max_pool2d(e3, 2))\n\n        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))\n        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))\n        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))\n        return self.head(d1)\n\n\ndef normalise(batch_bgr_float: torch.Tensor) -> torch.Tensor:\n    \"\"\"Normalise an (N, 3, H, W) RGB tensor already scaled to [0, 1].\"\"\"\n    mean = torch.tensor(MEAN, dtype=batch_bgr_float.dtype,\n                        device=batch_bgr_float.device).view(1, 3, 1, 1)\n    std = torch.tensor(STD, dtype=batch_bgr_float.dtype,\n                       device=batch_bgr_float.device).view(1, 3, 1, 1)\n    return (batch_bgr_float - mean) / std\n\n\ndef count_parameters(model: nn.Module) -> int:\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n", "train_card_seg.py": "\"\"\"Train the card segmenter on synthetic scenes. Written to finish on a CPU.\n\nRun it from the Colab notebook (which mounts Drive and points ``--photos`` at\n``data_set_pre/jpg``), or locally against any folder of single-card photos.\n\nScenes are generated once up front rather than on the fly: on two cores the\ncompositor is slower than the network, so caching a fixed set and re-augmenting\nit cheaply each epoch keeps the cores on the part that actually learns.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport random\nimport time\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\n\nimport card_synth as cs\nfrom tiny_unet import INPUT_H, INPUT_W, NUM_CLASSES, TinyUNet, count_parameters, normalise\n\nIMAGE_EXTS = {\".jpg\", \".jpeg\", \".png\", \".bmp\"}\n\n\nclass SceneDataset(Dataset):\n    \"\"\"Pre-generated scenes, re-augmented photometrically on every epoch.\"\"\"\n\n    def __init__(self, images: list[np.ndarray], labels: list[np.ndarray],\n                 train: bool):\n        self.images = images\n        self.labels = labels\n        self.train = train\n\n    def __len__(self) -> int:\n        return len(self.images)\n\n    def __getitem__(self, idx: int):\n        img = self.images[idx]\n        lab = self.labels[idx]\n\n        if self.train:\n            if random.random() < 0.5:\n                img, lab = cv2.flip(img, 1), cv2.flip(lab, 1)\n            if random.random() < 0.2:\n                img, lab = cv2.flip(img, 0), cv2.flip(lab, 0)\n            if random.random() < 0.7:\n                gain = random.uniform(0.75, 1.3)\n                bias = random.uniform(-22, 22)\n                img = np.clip(img.astype(np.float32) * gain + bias,\n                              0, 255).astype(np.uint8)\n\n        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0\n        x = torch.from_numpy(rgb).permute(2, 0, 1)\n        y = torch.from_numpy(lab.astype(np.int64))\n        return x, y\n\n\ndef build_dataset(photo_dir: Path, n_scenes: int, seed: int,\n                  max_photos: int | None) -> tuple[list, list]:\n    paths = sorted(p for p in photo_dir.rglob(\"*\")\n                   if p.suffix.lower() in IMAGE_EXTS)\n    if not paths:\n        raise SystemExit(f\"no images under {photo_dir}\")\n    print(f\"found {len(paths)} photos under {photo_dir}\")\n\n    print(\"extracting cards and background plates...\")\n    cards, plates = cs.load_sources(paths, limit=max_photos)\n    if not cards or not plates:\n        raise SystemExit(\"could not extract any card/plate pairs\")\n    print(f\"  -> {len(cards)} cards, {len(plates)} plates\")\n\n    print(f\"composing {n_scenes} scenes...\")\n    images, labels = [], []\n    rng = np.random.default_rng(seed)\n    t0 = time.time()\n    while len(images) < n_scenes:\n        scene = cs.compose_scene(cards, plates, rng, size=(INPUT_W, INPUT_H))\n        if not scene.cards:\n            continue\n        images.append(scene.image)\n        labels.append(scene.label_map(border_px=2))\n        if len(images) % 250 == 0:\n            print(f\"  {len(images)}/{n_scenes}  ({time.time() - t0:.0f}s)\")\n    return images, labels\n\n\ndef dice_loss(logits: torch.Tensor, target: torch.Tensor,\n              eps: float = 1.0) -> torch.Tensor:\n    \"\"\"Soft Dice over the two card classes.\n\n    Cross-entropy alone under-weights the border class — it is a couple of\n    percent of the pixels, yet it is the entire mechanism for pulling touching\n    cards apart. Dice scores each class by overlap rather than by pixel count, so\n    the thin border carries the same weight as the interior.\n    \"\"\"\n    probs = F.softmax(logits, dim=1)\n    onehot = F.one_hot(target, NUM_CLASSES).permute(0, 3, 1, 2).float()\n    dims = (0, 2, 3)\n    inter = (probs * onehot).sum(dims)\n    denom = probs.sum(dims) + onehot.sum(dims)\n    dice = (2 * inter + eps) / (denom + eps)\n    return 1.0 - dice[1:].mean()\n\n\n@torch.no_grad()\ndef evaluate(model: nn.Module, loader: DataLoader, device: torch.device,\n             class_weights: torch.Tensor) -> dict[str, float]:\n    model.eval()\n    inter = torch.zeros(NUM_CLASSES)\n    union = torch.zeros(NUM_CLASSES)\n    total_loss = 0.0\n    batches = 0\n    for x, y in loader:\n        x, y = x.to(device), y.to(device)\n        logits = model(normalise(x))\n        loss = (F.cross_entropy(logits, y, weight=class_weights)\n                + dice_loss(logits, y))\n        total_loss += float(loss)\n        batches += 1\n        pred = logits.argmax(1)\n        for c in range(NUM_CLASSES):\n            p, t = pred == c, y == c\n            inter[c] += float((p & t).sum())\n            union[c] += float((p | t).sum())\n    iou = (inter / union.clamp(min=1)).tolist()\n    return {\n        \"loss\": total_loss / max(batches, 1),\n        \"iou_background\": iou[0],\n        \"iou_interior\": iou[1],\n        \"iou_border\": iou[2],\n        \"miou_cards\": (iou[1] + iou[2]) / 2,\n    }\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--photos\", required=True,\n                    help=\"folder of single-card photos (searched recursively)\")\n    ap.add_argument(\"--out\", default=\"card_seg_unet.pt\")\n    ap.add_argument(\"--scenes\", type=int, default=3000)\n    ap.add_argument(\"--val-scenes\", type=int, default=300)\n    ap.add_argument(\"--epochs\", type=int, default=28)\n    ap.add_argument(\"--batch-size\", type=int, default=8)\n    ap.add_argument(\"--lr\", type=float, default=3e-3)\n    ap.add_argument(\"--width\", type=int, default=16)\n    ap.add_argument(\"--workers\", type=int, default=2)\n    ap.add_argument(\"--max-photos\", type=int, default=None)\n    ap.add_argument(\"--seed\", type=int, default=0)\n    args = ap.parse_args()\n\n    torch.manual_seed(args.seed)\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    if device.type == \"cpu\":\n        torch.set_num_threads(max(1, torch.get_num_threads()))\n    print(f\"device: {device}\")\n\n    tr_imgs, tr_labs = build_dataset(Path(args.photos), args.scenes,\n                                     args.seed, args.max_photos)\n    va_imgs, va_labs = build_dataset(Path(args.photos), args.val_scenes,\n                                     args.seed + 9999, args.max_photos)\n\n    train_loader = DataLoader(SceneDataset(tr_imgs, tr_labs, True),\n                              batch_size=args.batch_size, shuffle=True,\n                              num_workers=args.workers, drop_last=True)\n    val_loader = DataLoader(SceneDataset(va_imgs, va_labs, False),\n                            batch_size=args.batch_size, shuffle=False,\n                            num_workers=args.workers)\n\n    model = TinyUNet(width=args.width).to(device)\n    print(f\"parameters: {count_parameters(model):,}\")\n\n    # The border class is a sliver of the frame; without an explicit weight the\n    # optimiser happily ignores it and the model loses its ability to separate.\n    class_weights = torch.tensor([0.6, 1.0, 3.0], device=device)\n\n    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)\n    sched = torch.optim.lr_scheduler.OneCycleLR(\n        opt, max_lr=args.lr, total_steps=args.epochs * len(train_loader),\n        pct_start=0.25)\n\n    best = -1.0\n    for epoch in range(1, args.epochs + 1):\n        model.train()\n        t0 = time.time()\n        running = 0.0\n        for x, y in train_loader:\n            x, y = x.to(device), y.to(device)\n            logits = model(normalise(x))\n            loss = (F.cross_entropy(logits, y, weight=class_weights)\n                    + dice_loss(logits, y))\n            opt.zero_grad(set_to_none=True)\n            loss.backward()\n            opt.step()\n            sched.step()\n            running += float(loss.detach())\n\n        stats = evaluate(model, val_loader, device, class_weights)\n        print(f\"epoch {epoch:3d}/{args.epochs}  \"\n              f\"train_loss={running / max(len(train_loader), 1):.4f}  \"\n              f\"val_loss={stats['loss']:.4f}  \"\n              f\"IoU interior={stats['iou_interior']:.3f} \"\n              f\"border={stats['iou_border']:.3f}  \"\n              f\"({time.time() - t0:.0f}s)\")\n\n        if stats[\"miou_cards\"] > best:\n            best = stats[\"miou_cards\"]\n            save(model, args.out, args.width)\n            print(f\"  saved {args.out} (mIoU {best:.3f})\")\n\n    print(f\"done. best card mIoU = {best:.3f}\")\n\n\ndef save(model: nn.Module, path: str, width: int) -> None:\n    \"\"\"Export TorchScript so the service loads it without this training code.\"\"\"\n    model.eval()\n    cpu_model = TinyUNet(width=width)\n    cpu_model.load_state_dict({k: v.detach().cpu()\n                               for k, v in model.state_dict().items()})\n    cpu_model.eval()\n    example = torch.zeros(1, 3, INPUT_H, INPUT_W)\n    scripted = torch.jit.trace(cpu_model, example)\n    scripted = torch.jit.freeze(scripted)\n    torch.jit.save(scripted, path)\n\n\nif __name__ == \"__main__\":\n    main()\n", "card_seg_eval.py": "\"\"\"Score the trained model against the classical engine, split by layout.\n\nOne overall number would hide the thing that matters. The classical detector is\nalready strong on well-separated cards and helpless once they overlap, so the\nreport breaks results out by layout — a model that improves the average while\nregressing the separated case is not an improvement.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import Sequence\n\nimport cv2\nimport numpy as np\n\nimport card_synth as cs\n\nMODES = [\"row_gap\", \"row_touch\", \"row_overlap\", \"fan\"]\nMODE_LABEL_JA = {\n    \"row_gap\": \"離れて並ぶ\",\n    \"row_touch\": \"接触\",\n    \"row_overlap\": \"重なり\",\n    \"fan\": \"扇状（持ち手）\",\n}\nIOU_MATCH = 0.7\n\n\ndef _quad_iou(a: np.ndarray, b: np.ndarray, shape: tuple[int, int]) -> float:\n    h, w = shape\n    ma = np.zeros((h, w), np.uint8)\n    mb = np.zeros((h, w), np.uint8)\n    cv2.fillConvexPoly(ma, a.astype(np.int32), 1)\n    cv2.fillConvexPoly(mb, b.astype(np.int32), 1)\n    union = int(np.count_nonzero(ma | mb))\n    return int(np.count_nonzero(ma & mb)) / union if union else 0.0\n\n\ndef build_benchmark(cards: Sequence[np.ndarray], plates: Sequence[cs.Plate],\n                    per_mode: int = 30, seed: int = 20260803,\n                    size: tuple[int, int] = (720, 540)) -> dict:\n    \"\"\"Fixed scenes, so two engines are scored on exactly the same images.\"\"\"\n    rng = np.random.default_rng(seed)\n    bench: dict[str, list] = {}\n    for mode in MODES:\n        items = []\n        while len(items) < per_mode:\n            scene = cs.compose_scene(cards, plates, rng, size=size, mode=mode)\n            if not scene.cards:\n                continue\n            items.append((scene.image, [c.quad for c in scene.cards]))\n        bench[mode] = items\n    return bench\n\n\ndef score(bench: dict, detect) -> dict[str, dict[str, float]]:\n    out: dict[str, dict[str, float]] = {}\n    for mode, items in bench.items():\n        count_exact = matched = truth_total = pred_total = 0\n        for image, truth in items:\n            pred = detect(image)\n            truth_total += len(truth)\n            pred_total += len(pred)\n            if len(pred) == len(truth):\n                count_exact += 1\n            used: set[int] = set()\n            for t in truth:\n                best, best_j = 0.0, -1\n                for j, p in enumerate(pred):\n                    if j in used:\n                        continue\n                    v = _quad_iou(t, p.corners, image.shape[:2])\n                    if v > best:\n                        best, best_j = v, j\n                if best_j >= 0 and best >= IOU_MATCH:\n                    used.add(best_j)\n                    matched += 1\n        out[mode] = {\n            \"count_exact\": count_exact / max(len(items), 1),\n            \"recall\": matched / max(truth_total, 1),\n            \"precision\": matched / max(pred_total, 1),\n        }\n    return out\n\n\ndef _engines():\n    \"\"\"Import both engines, whether standing alone or inside the repo package.\n\n    On Colab the notebook drops every module into one directory, so a plain\n    import finds them. In the repo they live inside the recognition service's\n    package, which a plain import cannot reach — and this module is useful from\n    there too, for checking a downloaded model before it ships.\n    \"\"\"\n    try:\n        import card_seg_model  # type: ignore\n        import card_splitter_v2 as v2  # type: ignore\n        return card_seg_model, v2\n    except ImportError:\n        pass\n\n    import sys\n    from pathlib import Path\n\n    service = Path(__file__).resolve().parents[2] / \"services\" / \"recognition\"\n    if str(service) not in sys.path:\n        sys.path.insert(0, str(service))\n    from app.recognition import card_seg_model  # type: ignore\n    from app.recognition import card_splitter_v2 as v2  # type: ignore\n    return card_seg_model, v2\n\n\ndef report(cards: Sequence[np.ndarray], plates: Sequence[cs.Plate],\n           weights_path: str, per_mode: int = 30) -> dict:\n    \"\"\"Print a side-by-side comparison and return the raw numbers.\"\"\"\n    card_seg_model, v2 = _engines()\n\n    print(f\"ベンチマークを作成中（各レイアウト {per_mode} シーン）...\")\n    bench = build_benchmark(cards, plates, per_mode=per_mode)\n\n    card_seg_model.set_weights_path(weights_path)\n    results = {\n        \"classic\": score(bench, lambda im: v2.detect_cards_classic(im)),\n        \"learned\": score(bench, lambda im: card_seg_model.detect_cards_learned(im)),\n    }\n\n    header = (f\"{'レイアウト':<16}{'枚数一致':>18}{'検出率':>18}{'適合率':>18}\")\n    print(\"\\n\" + header)\n    print(f\"{'':<16}{'古典 → 学習':>20}{'古典 → 学習':>20}{'古典 → 学習':>20}\")\n    print(\"-\" * 74)\n    for mode in MODES:\n        c = results[\"classic\"][mode]\n        l = results[\"learned\"][mode]\n        print(f\"{MODE_LABEL_JA[mode]:<16}\"\n              f\"{c['count_exact']:>7.0%} → {l['count_exact']:<7.0%}\"\n              f\"{c['recall']:>9.0%} → {l['recall']:<7.0%}\"\n              f\"{c['precision']:>9.0%} → {l['precision']:<7.0%}\")\n\n    def mean(engine: str, key: str) -> float:\n        return float(np.mean([results[engine][m][key] for m in MODES]))\n\n    print(\"-\" * 74)\n    print(f\"{'平均':<16}\"\n          f\"{mean('classic', 'count_exact'):>7.0%} → {mean('learned', 'count_exact'):<7.0%}\"\n          f\"{mean('classic', 'recall'):>9.0%} → {mean('learned', 'recall'):<7.0%}\"\n          f\"{mean('classic', 'precision'):>9.0%} → {mean('learned', 'precision'):<7.0%}\")\n    print(\"\\n検出率 = 正解カードのうち IoU 0.7 以上で見つけられた割合\")\n    return results\n"}''')

for name, text in FILES.items():
    pathlib.Path('/content', name).write_text(text, encoding='utf-8')
    print('wrote', name, f'({len(text.splitlines())} lines)')

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

## 3. 写真からカードと背景を取り出す

各写真から「平らに直したカード」と「カードを消したテーブル」を取り出します。
1枚だけ写っている写真は検出が確実なので、ここは自動で通ります。
うまく検出できなかった写真は自動的に除外されます。

`PHOTO_DIR` が違う場合はここを書き換えてください。

In [ ]:
PHOTO_DIR = '/content/drive/MyDrive/data_set_pre/jpg'

import pathlib, cv2, numpy as np
import card_synth as cs

EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
paths = sorted(p for p in pathlib.Path(PHOTO_DIR).rglob('*')
               if p.suffix.lower() in EXTS)
print(f'写真: {len(paths)} 枚')
assert paths, f'画像が見つかりません: {PHOTO_DIR}'

cards, plates = cs.load_sources(paths)
print(f'\n取り出せたカード: {len(cards)} 枚 / 背景: {len(plates)} 枚')
assert cards and plates, 'カードを取り出せませんでした。PHOTO_DIR を確認してください。'

# 素材を学習用と評価用に分けます。同じカード・同じ背景で学習して評価すると
# 「見たことのある絵柄」を当てているだけになり、精度が実力より高く出ます。
rs = np.random.RandomState(0)
idx = rs.permutation(len(cards))
n_hold = max(1, int(len(cards) * 0.15))
cards_test = [cards[i] for i in idx[:n_hold]]
cards_train = [cards[i] for i in idx[n_hold:]]
pidx = rs.permutation(len(plates))
p_hold = max(1, int(len(plates) * 0.15))
plates_test = [plates[i] for i in pidx[:p_hold]]
plates_train = [plates[i] for i in pidx[p_hold:]]
print(f'学習用: カード{len(cards_train)} 背景{len(plates_train)} / 評価用(未使用): カード{len(cards_test)} 背景{len(plates_test)}')

In [ ]:
# 取り出した結果を目視確認
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(15, 6))
for i, ax in enumerate(axes[0]):
    ax.imshow(cv2.cvtColor(cards[i % len(cards)], cv2.COLOR_BGR2RGB))
    ax.set_title('card'); ax.axis('off')
for i, ax in enumerate(axes[1]):
    ax.imshow(cv2.cvtColor(plates[i % len(plates)].image, cv2.COLOR_BGR2RGB))
    ax.set_title('background'); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. 合成シーンを作る

カードを2〜5枚、ランダムに回転・遠近・**重なり**・光沢・影を付けて背景に配置します。
自分で配置しているので、正解は画素単位で完全に正確です。

In [ ]:
from tiny_unet import INPUT_W, INPUT_H

N_TRAIN = 3000   # CPUで重い場合は 1500 程度に下げてください
N_VAL = 300

import time
def make(n, seed):
    rng = np.random.default_rng(seed)
    ims, labs = [], []
    t0 = time.time()
    while len(ims) < n:
        s = cs.compose_scene(cards_train, plates_train, rng, size=(INPUT_W, INPUT_H))
        if not s.cards:
            continue
        ims.append(s.image); labs.append(s.label_map(border_px=2))
        if len(ims) % 500 == 0:
            print(f'  {len(ims)}/{n}  ({time.time()-t0:.0f}s)')
    return ims, labs

train_imgs, train_labs = make(N_TRAIN, 1)
val_imgs, val_labs = make(N_VAL, 2)
print(f'学習 {len(train_imgs)} / 検証 {len(val_imgs)}')

In [ ]:
# 合成シーンと正解ラベルを確認（黒=背景 / 灰=カード内部 / 白=境界）
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for i in range(4):
    axes[0][i].imshow(cv2.cvtColor(train_imgs[i], cv2.COLOR_BGR2RGB))
    axes[0][i].axis('off')
    axes[1][i].imshow(train_labs[i], vmin=0, vmax=2, cmap='gray')
    axes[1][i].axis('off')
plt.tight_layout(); plt.show()

## 5. 学習

GPUがあれば自動で使います。無い場合もCPUで動くようにモデルを小さく設計してあります
（約39万パラメータ）。時間がかかる場合は上の `N_TRAIN` と下の `EPOCHS` を下げてください。

In [ ]:
import torch, torch.nn.functional as F, random
from torch.utils.data import DataLoader
from tiny_unet import TinyUNet, count_parameters, normalise
import train_card_seg as T

EPOCHS = 28
BATCH = 8
LR = 3e-3

torch.manual_seed(0); random.seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

tl = DataLoader(T.SceneDataset(train_imgs, train_labs, True),
                batch_size=BATCH, shuffle=True, drop_last=True, num_workers=2)
vl = DataLoader(T.SceneDataset(val_imgs, val_labs, False),
                batch_size=BATCH, num_workers=2)

model = TinyUNet(width=16).to(device)
print(f'パラメータ数: {count_parameters(model):,}')

cw = torch.tensor([0.6, 1.0, 3.0], device=device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=LR, total_steps=EPOCHS * len(tl), pct_start=0.25)

best = -1.0
for ep in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); run = 0.0
    for x, y in tl:
        x, y = x.to(device), y.to(device)
        logits = model(normalise(x))
        loss = F.cross_entropy(logits, y, weight=cw) + T.dice_loss(logits, y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        run += float(loss)
    st = T.evaluate(model, vl, device, cw)
    print(f"ep{ep:3d}/{EPOCHS} train={run/len(tl):.4f} val={st['loss']:.4f} "
          f"IoU 内部={st['iou_interior']:.3f} 境界={st['iou_border']:.3f} "
          f'({time.time()-t0:.0f}s)')
    if st['miou_cards'] > best:
        best = st['miou_cards']
        T.save(model, '/content/card_seg_unet.pt', 16)
        print(f'  → 保存しました (mIoU {best:.3f})')
print(f'\n完了。ベスト mIoU = {best:.3f}')

## 6. 精度を確認する

**枚数がぴったり合った割合**と**1枚ごとの検出率**を、重なりの有無で分けて出します。
古典的手法（学習なし）と並べて比べられます。

In [ ]:
import card_splitter_v2 as v2
import card_seg_eval

# 学習に使っていないカード・背景だけでベンチマークを作って採点します。
card_seg_eval.report(cards_test, plates_test, '/content/card_seg_unet.pt')

## 7. 重みを Drive に保存

保存したら Drive からダウンロードして、リポジトリの
`services/recognition/models/card_seg_unet.pt` に置いてください。
その後 `fly deploy -a handhistory-recognition` でアプリに反映されます。

In [ ]:
import shutil, os
dest_dir = '/content/drive/MyDrive/hand_history_models'
os.makedirs(dest_dir, exist_ok=True)
dest = os.path.join(dest_dir, 'card_seg_unet.pt')
shutil.copy('/content/card_seg_unet.pt', dest)
print('保存しました:', dest)
print('サイズ:', round(os.path.getsize(dest) / 1e6, 2), 'MB')

from google.colab import files
files.download('/content/card_seg_unet.pt')